# FID and KID: Evaluating Generated Image Quality

---

**What is this notebook about?**

When we generate images with models like DDPM, how do we know if they're any good? We can look at them, but that's subjective and doesn't scale. We need **quantitative metrics**!

This notebook covers two important metrics for evaluating generated images:

1. **FID (Fréchet Inception Distance)**: Compares the distribution of generated images to real images
2. **KID (Kernel Inception Distance)**: Similar to FID but with some advantages

---

**Why do we need these metrics?**

| Method | Problem |
|--------|--------|
| Human inspection | Subjective, doesn't scale |
| Pixel comparison | Doesn't capture perceptual quality |
| Classification accuracy | Only tests if images are recognizable |

FID and KID solve these by comparing **feature distributions** - not raw pixels, but high-level representations learned by a neural network.

---

**The Key Idea:**

```
Real Images ──► Feature Extractor ──► Feature Distribution A
                                                              } Compare!
Generated Images ──► Feature Extractor ──► Feature Distribution B
```

If the generated images are good, their features should have a similar distribution to real images.

---

## Google Colab Setup

Run the cells below **once** at the start of each Colab session. They mount Google Drive, set the working directory, install required packages, and clone the `miniai` library from the fast.ai Part 2 course repo.

**GPU note:** FID requires running InceptionV3 over many images &mdash; GPU strongly recommended.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Fast.AI_Colab')
print(os.getcwd())

In [ ]:
!pip install -q fastcore fastai diffusers datasets torcheval accelerate
if not os.path.exists('course22p2'):
    !git clone https://github.com/fastai/course22p2.git
import sys
sys.path.insert(0, os.path.join(os.getcwd(), 'course22p2'))
try:
    import miniai
    print(f'miniai loaded from: {miniai.__file__}')
except ImportError:
    print('ERROR: miniai not found.')

In [ ]:
!pip install nbdev

---

*The cells above are Colab-specific setup. The content below is the same as the source notebook (`18_fid_explained.ipynb`), unchanged.*

---

---

## 🎬 Interactive Visualizations in This Notebook

This notebook includes **nine hands-on interactive visualizations** that you can click, drag, scrub, and play. Each one targets a concept that is much easier to *see* than to read about:

| # | Visualization | What it makes clear |
|---|---------------|---------------------|
| 1 | **FID/KID Grand Tour** | The whole story end to end — real & generated images race through the pipeline side by side; click any stage node to jump there |
| 2 | **3D Hook & Head-Chop Explorer** | Where features come from — a hook eavesdropping on layer 6 vs deleting the classifier head (drag to rotate) |
| 3 | **3D Gaussian Stats Lab** | μ as a ball, Σ as an ellipsoid, in real 3D — deform one cloud with sliders and watch the exact FID react |
| 4 | **Newton–Schulz Stepper** | The √M iteration, one step at a time — matrices, the bending ellipse, and a clickable error curve |
| 5 | **FID Formula Anatomy** | Click either term of the formula to isolate it: "centers apart" vs "shapes apart", each with its own slider and bar |
| 6 | **KID Kernel Arena** | Drag two point clouds, watch the three kernel matrices fill cell by cell, hover any cell for its exact arithmetic |
| 7 | **Bias–Variance Arena** | Rig both samples from the *same* distribution and catch FID lying (always > 0) while KID honestly straddles 0 |
| 8 | **Denoising Scoreboard** | FID & KID across all 1000 sampling steps — scrub the journey, click the curve, watch the thumbnail denoise |
| 9 | **Extractor Switch Lab** | Same images, two rulers: custom net says 33.8, InceptionV3 says 63.8 — and why you must never compare across rulers |

Every visualization has **clickable stages** (jump anywhere, the matching code lights up), **Play/Step controls**, and a **speed slider**; several add drag interactions, 3D orbit controls, or live exact math. Just run the cell beneath each section. *(They are self-contained HTML files — nothing to install.)*

## 🛠️ Interactive visualizations — one-time setup

The cells marked **🎮 Interactive** below embed standalone HTML/JS visualizations
(stored in the `interactive_viz/` folder next to this notebook) at **full width and
auto-fitted height**. Run the next cell once, then run any 🎮 cell.

In [ ]:
# ============================================================================
# INTERACTIVE VISUALIZATIONS -- setup (run this cell ONCE).
# Each "🎮 Interactive" cell below loads a standalone HTML file from the
# published copy on GitHub Pages, in an isolated <iframe> (its CSS/JS can't leak
# into the notebook). It is shown FULL WIDTH and auto-fits its content height.
# No local files needed -- the visualizations are read from GitHub.
# ============================================================================
from IPython.display import HTML

def show_viz(path, height="600px"):
    """Embed an interactive visualization full width; it auto-fits its height.
    `path` may be a bare 'interactive_viz/<file>.html' (resolved to the GitHub
    Pages copy) or a full https URL."""
    base = "https://shammun.github.io/shammunul-fastai-notes/notebooks/"
    if not path.startswith("http"):
        path = base + path
    return HTML(
        f'<iframe src="{path}" loading="lazy" allowfullscreen '
        f'style="width:100%;height:{height};border:1px solid #dde5f2;border-radius:12px;'
        f'box-shadow:0 8px 24px rgba(123,92,214,.12);background:#fff;"></iframe>'
        '<script>addEventListener("message",function(e){'
        'if(e.data&&e.data.type==="ce-frame-height"&&e.data.height>50){'
        'var fs=document.querySelectorAll("iframe");for(var i=0;i<fs.length;i++){'
        'if(fs[i].contentWindow===e.source){fs[i].style.height=e.data.height+"px";break;}}}});</script>'
    )

### 🎮 Interactive: the grand tour — the whole FID/KID pipeline in one picture

Before any code, get the bird's-eye view. **Real images** (top lane) and **generated images** (bottom lane) each flow through the *same frozen feature extractor*, become clouds of 512-D points, get squashed to (μ, Σ), and finally meet in one number. Every later section of this notebook zooms into exactly one node of this picture.

**What to try:**
- Press **▶ Play all** and watch both lanes advance in lock-step — the fairness of FID lives in that symmetry.
- **Click any stage node on the canvas** (or its chip above) to jump straight there; the code panel below switches to the exact notebook lines for that stage.
- Flip the **quality dropdown** between *bad / this notebook (FID ≈ 33.8) / great* and watch only the generated lane's cloud move — the real lane never changes.
- Use the **speed slider** if the tour moves too fast or too slow.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/fid_grand_tour.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/fid_grand_tour.html", height="760px")

In [ ]:
#| default_exp fid
# This tells nbdev to export marked cells to the 'fid' module

## Section 1: Imports and Setup

In [ ]:
#|export
# ============================================================================
# STANDARD IMPORTS
# ============================================================================

import pickle, gzip, math, os, time, shutil, torch, random
import fastcore.all as fc
import matplotlib as mpl
import numpy as np
import matplotlib.pyplot as plt

from collections.abc import Mapping
from pathlib import Path
from operator import attrgetter, itemgetter
from functools import partial
from copy import copy
from contextlib import contextmanager
from scipy import linalg  # For matrix square root

from fastcore.foundation import L
import torchvision.transforms.functional as TF
import torch.nn.functional as F

from torch import tensor, nn, optim
from torch.utils.data import DataLoader, default_collate
from torch.nn import init
from torch.optim import lr_scheduler
from torcheval.metrics import MulticlassAccuracy
from datasets import load_dataset, load_dataset_builder

# ============================================================================
# MINIAI IMPORTS
# ============================================================================

from miniai.datasets import *
from miniai.conv import *
from miniai.learner import *
from miniai.activations import *
from miniai.init import *
from miniai.sgd import *
from miniai.resnet import *
from miniai.augment import *
from miniai.accel import *

In [ ]:
from fastcore.test import test_close
from torch import distributions

# Configure torch printing for cleaner output
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
torch.manual_seed(1)

# Reversed grayscale colormap (dark clothes on light background)
mpl.rcParams['image.cmap'] = 'gray_r'

# Suppress warning messages
import logging
logging.disable(logging.WARNING)

# Set random seed for reproducibility
set_seed(42)

# Limit CPU workers if there are many
if fc.defaults.cpus > 8:
    fc.defaults.cpus = 8

**What does the code above do?**

Standard imports plus:
- `scipy.linalg`: For computing matrix square root (needed for FID)
- `torch.set_printoptions`: Makes tensor output easier to read
- Seed setting for reproducibility

---

## Section 2: The Feature Extractor (Classifier)

FID and KID work by comparing **features** (not raw pixels). We need a neural network to extract these features.

**Option 1**: Use a classifier trained on our dataset (Fashion MNIST)
**Option 2**: Use InceptionV3 (the standard for FID)

Let's start with Option 1 - using our own trained classifier.

In [ ]:
# ============================================================================
# LOAD FASHION MNIST DATASET
# ============================================================================

xl, yl = 'image', 'label'  # Column names
name = "fashion_mnist"
bs = 512  # Batch size

@inplace
def transformi(b):
    """
    Transform images:
    1. Convert to tensor
    2. Pad from 28x28 to 32x32
    3. Scale from [0,1] to [-1,1]: multiply by 2, subtract 1
    
    The [-1,1] range matches what our models expect.
    """
    # F.pad adds 2 pixels on each side: (left, right, top, bottom)
    # *2-1 transforms [0,1] to [-1,1]
    b[xl] = [F.pad(TF.to_tensor(o), (2, 2, 2, 2)) * 2 - 1 for o in b[xl]]

# Load and transform dataset
dsd = load_dataset(name)
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=fc.defaults.cpus)

**What does the code above do?**

Loads Fashion MNIST with preprocessing:
- **Padding**: 28×28 → 32×32
- **Normalization**: [0,1] → [-1,1]

The `*2-1` transformation:
```
0.0 → 0.0*2-1 = -1.0
0.5 → 0.5*2-1 =  0.0
1.0 → 1.0*2-1 =  1.0
```

In [ ]:
# Get a batch of data
b = xb, yb = next(iter(dls.train))

In [ ]:
# Load a pre-trained classifier
# This model was trained in a previous notebook on Fashion MNIST
# Look at 14_augment.ipynb for the model training code
cbs = [DeviceCB(), MixedPrecision()]
model = torch.load('models/data_aug2.pkl')
# We are using the model saved in 14_augment.ipynb

# Create learner (opt_func=None because we're not training)
learn = Learner(model, dls, F.cross_entropy, cbs=cbs, opt_func=None)

**What does the code above do?**

Loads a pre-trained Fashion MNIST classifier. We'll use this to extract features.

**Why use a classifier for features?**

A classifier trained to recognize clothing items has learned meaningful representations:
- Early layers: edges, textures
- Middle layers: patterns, parts
- Late layers: high-level features useful for classification

We'll extract features from a late layer (before the final classification head).

### Extracting Features with Hooks

We need to capture the output of an intermediate layer. PyTorch **hooks** let us do this!

In [ ]:
def append_outp(hook, mod, inp, outp):
    """
    Hook function that captures layer outputs.
    
    Parameters:
    -----------
    hook : Hook object
        The hook object (we'll store outputs here)
    mod : nn.Module
        The module this hook is attached to
    inp : tuple
        Input to the module (not used)
    outp : tensor
        Output from the module (this is what we want!)
    """
    # Initialize output list if needed
    if not hasattr(hook, 'outp'):
        hook.outp = []
    # Store the output (move to CPU to save GPU memory)
    hook.outp.append(to_cpu(outp))

**What does the code above do?**

Creates a hook function that captures layer outputs:

```
Input ──► Layer ──► Output
                      │
                      └──► Hook captures this!
```

**How hooks work:**
1. Register a hook on a layer
2. When that layer runs, PyTorch calls your hook function
3. Your function receives the layer's input and output
4. You can store, modify, or analyze the output

In [ ]:
# Create a HooksCallback to capture outputs from layer 6 (a late layer)
# on_valid=True means run during validation (when we're extracting features)
hcb = HooksCallback(append_outp, mods=[learn.model[6]], on_valid=True)
# We want to hook into the output of layer 6, which is the last layer before the final classification layer.
# This is because we want to extract features from the model, not the final predictions.

**What does the code above do?**

Creates a callback that:
1. Attaches our `append_outp` hook to layer 6 of the model
2. Only runs during validation (`on_valid=True`)

**Why layer 6?**

This is typically a late layer in the network (before the classifier head) that contains high-level features.

In [ ]:
# Run validation to capture features
# train=False means only run validation
learn.fit(1, train=False, cbs=[hcb])
# We run this to grab the outputs

In [ ]:
# Extract the captured features
# hcb.hooks[0] is the first (and only) hook
# .outp[0] is the first batch of outputs
# .float() converts to float32
# [:64] takes first 64 samples
feats = hcb.hooks[0].outp[0].float()[:64]
feats.shape

**What does the output show?**

`torch.Size([64, 512])` means:
- 64 images
- 512-dimensional feature vector for each image

Each image is now represented by 512 numbers that capture its high-level characteristics!

# Understanding PyTorch Hooks: A Deep Dive into HooksCallback and Feature Extraction

## Question

For this code block, inside the function, `mod` and `inp` parameters are never used, then why do we even have these and why do we need these and how these unused arguments help?

```python
def append_outp(hook, mod, inp, outp):
    """
    Hook function that captures layer outputs.
    
    Parameters:
    -----------
    hook : Hook object
        The hook object (we'll store outputs here)
    mod : nn.Module
        The module this hook is attached to
    inp : tuple
        Input to the module (not used)
    outp : tensor
        Output from the module (this is what we want!)
    """
    # Initialize output list if needed
    if not hasattr(hook, 'outp'):
        hook.outp = []
    # Store the output (move to CPU to save GPU memory)
    hook.outp.append(to_cpu(outp))
```

---

## Part 1: The Short Answer

The `mod` and `inp` parameters exist because **PyTorch's hook API always passes all arguments** - it's a fixed contract. Your function signature must match, or Python raises a TypeError. Even if you don't use them, you must accept them.

---

## Part 2: Context - Why Do We Need Hooks?

In the FID (Fréchet Inception Distance) notebook, we need to extract **intermediate features** from a neural network - not the final output, but the activations from a specific layer inside the model.

**The Problem:**
```
Input ──► Layer 0 ──► Layer 1 ──► ... ──► Layer 6 ──► Layer 7 ──► Layer 8 ──► Output
                                              │
                                              └── We want THIS! (512-dim features)
```

Normally, a neural network only gives you the final output. But for FID/KID, we need features from an intermediate layer (e.g., layer 6) because these capture high-level semantic information about the image.

**The Solution: PyTorch Hooks**

Hooks let us "tap into" any layer and capture what flows through it:

```
Input ──► Layer ──► Output
                      │
                      └──► Hook captures this!
```

---

## Part 3: How Does `learn.model[6]` Work?

### Understanding Model Indexing

In PyTorch/fastai, a model is often a `nn.Sequential` container - essentially a list of layers. You can index into it like a Python list:

```python
# A typical model structure
model = nn.Sequential(
    nn.Conv2d(...),      # model[0]
    nn.BatchNorm2d(...), # model[1]
    nn.ReLU(),           # model[2]
    nn.Conv2d(...),      # model[3]
    nn.BatchNorm2d(...), # model[4]
    nn.ReLU(),           # model[5]
    nn.AdaptiveAvgPool2d(...), # model[6]  <-- We hook HERE!
    nn.Flatten(),        # model[7]
    nn.Linear(...)       # model[8] - classification head
)
```

### When We Write:

```python
hcb = HooksCallback(append_outp, mods=[learn.model[6]], on_valid=True)
```

We're saying:
- `learn.model[6]` → Get the 7th layer (index 6) from the model
- `mods=[...]` → Create hooks on these specific modules
- `on_valid=True` → Only capture during validation (not training)

**Layer 6 is typically a late layer** (like AdaptiveAvgPool or a late conv block) that contains high-level, semantically meaningful features - perfect for FID/KID calculations!

---

## Part 4: The Hook Architecture - Three Key Classes

Understanding the hook system requires knowing three interconnected classes:

### 1. `Hook` Class (wraps a single PyTorch hook)

```python
class Hook():
    "Create a hook on `m` with `hook_func`."
    def __init__(self, m, hook_func, is_forward=True, detach=True, cpu=False, gather=False):
        self.hook_func = hook_func
        self.detach = detach
        # Register the hook with PyTorch!
        f = m.register_forward_hook if is_forward else m.register_backward_hook
        self.hook = f(self.hook_fn)  # <-- This is the key line!
        self.stored = None
        self.removed = False

    def hook_fn(self, module, input, output):
        "Called by PyTorch when the layer runs forward pass."
        if self.detach:
            input = to_detach(input, cpu=self.cpu)
            output = to_detach(output, cpu=self.cpu)
        # Call the user's hook function, passing self (the Hook) as first arg!
        self.stored = self.hook_func(self, module, input, output)

    def remove(self):
        if not self.removed:
            self.hook.remove()
            self.removed = True
```

**Key insight**: The `Hook` class:
1. Calls `m.register_forward_hook(self.hook_fn)` to register with PyTorch
2. When the layer runs, PyTorch calls `hook_fn(module, input, output)`
3. `hook_fn` then calls YOUR function with `(self, module, input, output)`

**This is why `append_outp` receives 4 arguments**: The `Hook` class adds itself as the first argument!

### 2. `Hooks` Class (manages multiple Hook objects)

```python
class Hooks():
    "Create several hooks on the modules in `ms` with `hook_func`."
    def __init__(self, ms, hook_func, is_forward=True, detach=True, cpu=False):
        # Create one Hook for each module in the list
        self.hooks = [Hook(m, hook_func, is_forward, detach, cpu) for m in ms]

    def __getitem__(self, i): return self.hooks[i]
    def __len__(self): return len(self.hooks)
    def __iter__(self): return iter(self.hooks)
    
    def remove(self):
        for h in self.hooks: h.remove()
```

### 3. `HooksCallback` Class (integrates with fastai's training loop)

```python
class HooksCallback(Callback):
    "Callback that registers hooks on `modules`."
    def __init__(self, hookfunc, mods=None, on_train=True, on_valid=True, ...):
        self.hookfunc = hookfunc
        self.mods = mods
        self.on_train = on_train
        self.on_valid = on_valid

    def before_fit(self):
        "Register the hooks before training/validation starts."
        self.hooks = Hooks(self.mods, self.hookfunc, ...)

    def after_fit(self):
        "Remove hooks after training/validation ends."
        self.hooks.remove()
```

---

## Part 5: Step-by-Step Execution Flow

Let's trace exactly what happens when you run this code:

### Step 1: Create the HooksCallback

```python
hcb = HooksCallback(append_outp, mods=[learn.model[6]], on_valid=True)
```

**What happens:**
```
┌─────────────────────────────────────────────────────────┐
│ HooksCallback.__init__() runs:                          │
│   • self.hookfunc = append_outp  (stores your function) │
│   • self.mods = [learn.model[6]] (stores layer 6)       │
│   • self.on_valid = True         (only run on valid)    │
│   • Hooks NOT created yet!                              │
└─────────────────────────────────────────────────────────┘
```

### Step 2: Start Validation

```python
learn.fit(1, train=False, cbs=[hcb])
```

**What happens when fit() begins:**
```
┌─────────────────────────────────────────────────────────────────┐
│ HooksCallback.before_fit() is called automatically:             │
│                                                                 │
│   self.hooks = Hooks(self.mods, self.hookfunc, ...)             │
│                  │                                              │
│                  ▼                                              │
│   Hooks.__init__() runs:                                        │
│     for m in [learn.model[6]]:                                  │
│       Hook(m, append_outp, ...)                                 │
│             │                                                   │
│             ▼                                                   │
│   Hook.__init__() runs:                                         │
│     self.hook_func = append_outp                                │
│     self.hook = m.register_forward_hook(self.hook_fn)           │
│                 │                                               │
│                 ▼                                               │
│   PyTorch now knows: "When layer 6 runs, call hook_fn()"        │
└─────────────────────────────────────────────────────────────────┘
```

### Step 3: Forward Pass Reaches Layer 6

When validation runs and data flows through the model:

```
┌─────────────────────────────────────────────────────────────────┐
│ Image batch flows through model:                                │
│                                                                 │
│   Input ──► Layer 0 ──► Layer 1 ──► ... ──► Layer 5             │
│                                                 │               │
│                                                 ▼               │
│                                            Layer 6              │
│                                                 │               │
│   ┌─────────────────────────────────────────────┴───────────┐   │
│   │ PyTorch executes layer 6's forward(), then:             │   │
│   │                                                         │   │
│   │ for hook in layer6._forward_hooks:                      │   │
│   │     hook(module, input, output)                         │   │
│   │           │       │       │                             │   │
│   │           ▼       ▼       ▼                             │   │
│   │     Hook.hook_fn(module, input, output) is called       │   │
│   └─────────────────────────────────────────────────────────┘   │
│                                                 │               │
│                                                 ▼               │
│                                            Layer 7 ──► ...      │
└─────────────────────────────────────────────────────────────────┘
```

### Step 4: Hook.hook_fn() Executes

```
┌─────────────────────────────────────────────────────────────────┐
│ Hook.hook_fn(self, module, input, output) runs:                 │
│                                                                 │
│   def hook_fn(self, module, input, output):                     │
│       # Detach tensors from computation graph                   │
│       input = to_detach(input)                                  │
│       output = to_detach(output)                                │
│                                                                 │
│       # Call YOUR function with Hook object as first arg!       │
│       self.stored = self.hook_func(self, module, input, output) │
│                                    │     │       │      │       │
│                                    ▼     ▼       ▼      ▼       │
│                          append_outp(hook, mod,  inp,  outp)    │
└─────────────────────────────────────────────────────────────────┘
```

### Step 5: Your append_outp() Function Runs

```python
def append_outp(hook, mod, inp, outp):
    #              │     │    │    │
    #              │     │    │    └── The 512-dim feature tensor!
    #              │     │    └─────── Input to layer 6 (unused)
    #              │     └──────────── Layer 6 module itself (unused)
    #              └────────────────── The Hook object (for storage!)
    
    # First call? Initialize the list
    if not hasattr(hook, 'outp'):
        hook.outp = []
    
    # Store the output features!
    hook.outp.append(to_cpu(outp))
```

**Line-by-line execution:**

| Line | What Happens |
|------|--------------|
| `if not hasattr(hook, 'outp'):` | Check if hook.outp exists (first batch: NO) |
| `hook.outp = []` | Create empty list on the Hook object |
| `hook.outp.append(to_cpu(outp))` | Move output tensor to CPU, add to list |

### Step 6: Repeat for Each Batch

For each validation batch:
```
Batch 1: hook.outp = [tensor_batch_1]          # shape: [512, 512]
Batch 2: hook.outp = [tensor_batch_1, tensor_batch_2]
Batch 3: hook.outp = [tensor_batch_1, tensor_batch_2, tensor_batch_3]
...
Batch N: hook.outp = [all N batch tensors]
```

### Step 7: Extract the Features

```python
feats = hcb.hooks[0].outp[0].float()[:64]
#       │         │    │    │        │
#       │         │    │    │        └── Take first 64 samples
#       │         │    │    └─────────── Convert to float32
#       │         │    └──────────────── First batch of outputs
#       │         └───────────────────── First (and only) Hook
#       └─────────────────────────────── HooksCallback's Hooks container
```

**Breaking it down:**

| Expression | What It Returns |
|------------|-----------------|
| `hcb` | The HooksCallback object |
| `hcb.hooks` | The Hooks container (holds all Hook objects) |
| `hcb.hooks[0]` | The first Hook object (attached to layer 6) |
| `hcb.hooks[0].outp` | List of all captured outputs (one per batch) |
| `hcb.hooks[0].outp[0]` | First batch's output tensor |
| `.float()` | Convert from float16/bfloat16 to float32 |
| `[:64]` | Take only first 64 samples |

**Result:** `torch.Size([64, 512])` - 64 images, each with 512 features!

---

## Part 6: Why Are `mod` and `inp` Required?

### The Technical Reason

PyTorch's `register_forward_hook` API has a **fixed signature**:

```python
# PyTorch internally does this:
for hook_fn in module._forward_hooks.values():
    hook_fn(module, input, output)  # ALWAYS passes all 3!
```

The fastai `Hook` wrapper then prepends itself:
```python
your_function(self, module, input, output)  # ALWAYS passes all 4!
```

**If your function doesn't accept all arguments:**
```python
# ❌ This crashes!
def bad_hook(hook, outp):
    hook.outp.append(outp)
    
# TypeError: bad_hook() takes 2 positional arguments but 4 were given
```

### Common Patterns for Unused Args

```python
# Option 1: Name them but don't use (the notebook does this)
def append_outp(hook, mod, inp, outp):
    hook.outp.append(outp)

# Option 2: Use underscore convention to signal "unused"
def append_outp(hook, _mod, _inp, outp):
    hook.outp.append(outp)

# Option 3: Use *args for truly don't-care args
def append_outp(hook, *args):
    outp = args[-1]  # output is always last
    hook.outp.append(outp)
```

---

## Part 7: When Would You Actually Use `mod` and `inp`?

While `append_outp` doesn't use them, these parameters are valuable in other scenarios:

### Using `mod` (module) for Debugging

```python
def debug_hook(hook, mod, inp, outp):
    print(f"Layer: {mod.__class__.__name__}")
    print(f"Output shape: {outp.shape}")
    print(f"Output mean: {outp.mean():.4f}")
    hook.outp.append(outp)
```

### Using `inp` (input) for Gradient Analysis

```python
def capture_both(hook, mod, inp, outp):
    # Store both input and output for later analysis
    if not hasattr(hook, 'inp'): hook.inp = []
    if not hasattr(hook, 'outp'): hook.outp = []
    hook.inp.append(to_cpu(inp[0]))   # inp is a tuple!
    hook.outp.append(to_cpu(outp))
```

### Using `mod` for Conditional Capture

```python
def selective_hook(hook, mod, inp, outp):
    # Only capture from Conv2d layers
    if isinstance(mod, nn.Conv2d):
        hook.outp.append(to_cpu(outp))
```

---

## Part 8: Visual Summary

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                          COMPLETE HOOK FLOW                                  │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  1. SETUP                                                                   │
│     hcb = HooksCallback(append_outp, mods=[learn.model[6]], on_valid=True)  │
│                              │                  │                           │
│                              ▼                  ▼                           │
│                        Your function      The layer to hook                 │
│                                                                             │
│  2. REGISTRATION (when fit() starts)                                        │
│     learn.model[6].register_forward_hook(Hook.hook_fn)                      │
│                                                                             │
│  3. EXECUTION (for each batch during validation)                            │
│                                                                             │
│     Image Batch                                                             │
│         │                                                                   │
│         ▼                                                                   │
│     ┌─────────┐    ┌─────────┐         ┌─────────┐                         │
│     │ Layer 0 │───►│ Layer 1 │───► ... │ Layer 5 │                         │
│     └─────────┘    └─────────┘         └────┬────┘                         │
│                                             │                               │
│                                             ▼                               │
│                                      ┌─────────────┐                        │
│                                      │   Layer 6   │                        │
│                                      │  (hooked!)  │                        │
│                                      └──────┬──────┘                        │
│                                             │                               │
│                    ┌────────────────────────┼────────────────────────┐      │
│                    │                        │                        │      │
│                    ▼                        ▼                        ▼      │
│              hook_fn called           Output continues          PyTorch     │
│                    │                   to Layer 7, 8...         internal    │
│                    ▼                                                        │
│         append_outp(hook, mod, inp, outp)                                   │
│                    │                                                        │
│                    ▼                                                        │
│            hook.outp.append(outp)                                           │
│                                                                             │
│  4. EXTRACTION (after validation)                                           │
│     feats = hcb.hooks[0].outp[0].float()[:64]                               │
│                                                                             │
│     Result: torch.Size([64, 512])                                           │
│             └─ 64 images, 512 features each                                 │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

---

## Part 9: The Analogy

Think of hooks like **security cameras** in a building:

1. **Installing the camera** (`register_forward_hook`) - You place a camera at a specific location (layer 6)
2. **Camera's fixed output** - The camera ALWAYS records: timestamp, location, and footage (module, input, output)
3. **Your monitoring function** (`append_outp`) - You receive ALL the data from the camera, but you might only care about the footage (output)
4. **Reviewing recordings** (`hcb.hooks[0].outp`) - Later, you look at what the camera captured

You can't tell the camera "only send me the footage" - it always sends everything. You just ignore what you don't need.

---

## Summary

| Question | Answer |
|----------|--------|
| Why do we need `mod` and `inp`? | PyTorch's hook API always passes all arguments - it's a fixed contract |
| How does `learn.model[6]` work? | Models are `nn.Sequential` containers; you can index layers like a list |
| What does `HooksCallback` do? | Wraps fastai's `Hooks` class to integrate with the training loop |
| When is `append_outp` called? | Every time Layer 6's forward pass completes |
| Where are outputs stored? | On the `Hook` object itself (`hook.outp`) |
| Why use `hook.outp` not a global? | Each Hook has its own storage; cleaner and allows multiple hooks |

The key insight is that **the Hook object acts as both the registrar and the storage container** - it registers with PyTorch AND provides a place to store captured data. That's why it passes itself as the first argument to your function!

### Modifying the Model for Feature Extraction
#### Alternative to Hooks

Instead of using hooks, we can also modify the model to output features directly by removing the final classification layers.

In [ ]:
# Remove the final layers (classifier head)
# Layer 8: final layer or BatchNorm1d
# Layer 7: linear layer
del(learn.model[8])
del(learn.model[7])

**What does the code above do?**

Removes the classification head from the model:

```
Before:                           After:
Input ──► Layers 0-6 ──► Layer 7 ──► Layer 8 ──► Class probabilities
                                                        
Input ──► Layers 0-6 ──► Features (512-dim)
```

Now the model outputs features instead of class predictions!

In [ ]:
# Extract features for the entire validation set
# capture_preds() runs the model and captures predictions
feats, y = learn.capture_preds()
feats = feats.float()  # Convert to float32

# Check shapes
feats.shape, y

(torch.Size([10000, 512]), tensor([9, 2, 1,  ..., 8, 1, 5]))

10000 images in the test set. Now, we have got what do 1000 real images look like at the end of the pooling layer.

So, now we need to do the same for the generated images or samples.

**What does the output show?**

- `feats.shape`: (10000, 512) - 10,000 images, each with 512 features
- `y`: The labels (0-9 for Fashion MNIST classes)

We now have feature vectors for all validation images!

### 🎮 Interactive: 3D tour — where do the 512 features come from?

You've now seen **both** ways to tap the classifier: a *hook* that eavesdrops on layer 6, and simply *deleting* layers 8 and 7 so the network ends where the features live. This 3D model shows both on the real `learn.model` layer stack.

**What to try**
- **Drag to rotate**, scroll to zoom. Click any layer slab to inspect its output shape.
- Toggle **Hook mode** — watch a probe attach to layer 6 while the full stack keeps running — vs **Chop mode**, where layers 7–8 fade away and the network's *new* output is the 512-vector.
- Press ▶ Play to send a batch flowing through; the pulse stops where features are captured.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/hook_feature_extractor.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/hook_feature_extractor.html", height="780px")

---

## Section 3: Calculating FID

Now let's understand and calculate FID!

**FID (Fréchet Inception Distance)** measures how similar two distributions of features are.

---

### The Math Behind FID

FID assumes features follow a multivariate Gaussian distribution. It compares:
- **Real image features**: mean μ₁, covariance Σ₁
- **Generated image features**: mean μ₂, covariance Σ₂

**The FID formula:**

$$FID = ||μ_1 - μ_2||^2 + Tr(Σ_1 + Σ_2 - 2\sqrt{Σ_1 Σ_2})$$

Where:
- $||μ_1 - μ_2||^2$ = squared distance between means
- $Tr()$ = trace (sum of diagonal elements)
- $\sqrt{Σ_1 Σ_2}$ = matrix square root of the product

**Intuition:**
- If means are different → higher FID
- If covariances are different → higher FID
- FID = 0 means identical distributions
- Lower FID = better generated images

### Generate Some Images to Evaluate

First, let's generate some images with our trained DDPM model.

In [ ]:
# Set up the noise schedule (same as DDPM training)
betamin, betamax, n_steps = 0.0001, 0.02, 1000
beta = torch.linspace(betamin, betamax, n_steps)
alpha = 1. - beta
alphabar = alpha.cumprod(dim=0)
sigma = beta.sqrt()

In [ ]:
def noisify(x0, alphabar_schedule):
    """
    Add noise to images (forward diffusion).
    Same as DDPM notebooks.
    """
    device = x0.device
    n = len(x0)
    t = torch.randint(0, n_steps, (n,), dtype=torch.long)
    epsilon = torch.randn(x0.shape, device=device)
    alphabar_t = alphabar_schedule[t].reshape(-1, 1, 1, 1).to(device)
    xt = alphabar_t.sqrt() * x0 + (1 - alphabar_t).sqrt() * epsilon
    return (xt, t.to(device)), epsilon


def collate_ddpm(b):
    """Custom collate function for DDPM."""
    return noisify(default_collate(b)[xl], alphabar)


def dl_ddpm(ds):
    """Create a DDPM DataLoader."""
    return DataLoader(ds, batch_size=bs, collate_fn=collate_ddpm, num_workers=4)

In [ ]:
# Create DDPM DataLoaders
dls2 = DataLoaders(dl_ddpm(tds['train']), dl_ddpm(tds['test']))

In [ ]:
# Import the UNet model from diffusers
from diffusers import UNet2DModel

class UNet(UNet2DModel):
    """Wrapper to accept tuple input and extract .sample."""
    def forward(self, x):
        return super().forward(*x).sample

In [ ]:
# Load the trained DDPM model
smodel = torch.load('models/fashion_ddpm_mp.pkl').cuda()

In [ ]:
@torch.no_grad()
def sample(model, sz, alpha, alphabar, sigma, n_steps):
    """
    Generate images using the DDPM sampling process.
    
    Same as previous notebooks, but saves x_0_hat (the clean image estimate)
    at each step instead of x_t.
    """
    device = next(model.parameters()).device
    x_t = torch.randn(sz, device=device)
    preds = []
    
    for t in reversed(range(n_steps)):
        t_batch = torch.full((x_t.shape[0],), t, device=device, dtype=torch.long)
        z = (torch.randn(x_t.shape) if t > 0 else torch.zeros(x_t.shape)).to(device)
        
        alphabar_t1 = alphabar[t-1] if t > 0 else torch.tensor(1)
        b_bar_t = 1 - alphabar[t]
        b_bar_t1 = 1 - alphabar_t1
        
        # Estimate x_0 from current x_t
        x_0_hat = ((x_t - b_bar_t.sqrt() * model((x_t, t_batch))) / alphabar[t].sqrt())
        
        # DDPM update step
        x_t = (x_0_hat * alphabar_t1.sqrt() * (1 - alpha[t]) / b_bar_t + 
               x_t * alpha[t].sqrt() * b_bar_t1 / b_bar_t + 
               sigma[t] * z)
        
        # Save x_0_hat (the estimated clean image) for visualization
        preds.append(x_0_hat.cpu())
    
    return preds

In [ ]:
%%time
# Generate 256 samples
# This takes about 30-40 seconds
samples = sample(smodel, (256, 1, 32, 32), alpha, alphabar, sigma, n_steps)

In [ ]:
# Get the final samples and scale to [-1, 1]
# samples[-1] is the last (most denoised) step
# *2-1 scales from [0,1] to [-1,1] to match our training data
s = samples[-1] * 2 - 1

In [ ]:
# Display some generated images
show_images(s[:16], imsize=1.5)

**What you should see:**

16 generated Fashion MNIST images. They should look like clothing items (shirts, pants, shoes, etc.).

### Extract Features from Generated Images

In [ ]:
# Create a learner to extract features from generated images
# DataLoaders([],[(s,yb)]): empty train loader, validation has our generated samples
clearn = TrainLearner(
    model,                          # Our feature extractor
    DataLoaders([], [(s, yb)]),     # Generated images as "validation" set # Dataloader with one batch
    # s is the final denoised output
    loss_func=fc.noop,              # No loss needed
    cbs=[DeviceCB()],
    opt_func=None                   # No optimizer needed
)
# We are going to use this to extract features from a model

# Extract features
feats2, y2 = clearn.capture_preds()
feats2 = feats2.float().squeeze()  # Remove extra dimensions
feats2.shape

**torch.size([256, 512])**

# Understanding Feature Extraction with TrainLearner and capture_preds()

## The Code in Question

```python
# Create a learner to extract features from generated images
# DataLoaders([],[(s,yb)]): empty train loader, validation has our generated samples
clearn = TrainLearner(
    model,                          # Our feature extractor
    DataLoaders([], [(s, yb)]),     # Generated images as "validation" set
    loss_func=fc.noop,              # No loss needed
    cbs=[DeviceCB()],
    opt_func=None                   # No optimizer needed
)

# Extract features
feats2, y2 = clearn.capture_preds()
feats2 = feats2.float().squeeze()  # Remove extra dimensions
feats2.shape
```

Output: `torch.Size([256, 512])`

---

## The Big Picture

We want to **extract features from generated images** using the same feature extractor model we used for real images. To do this, we're "tricking" the fastai learner into running inference on our generated samples.

---

## Part 1: Understanding `DataLoaders([], [(s, yb)])`

### What is a DataLoader?

A `DataLoaders` object holds two things:
1. **Training data** (first argument)
2. **Validation data** (second argument)

### Breaking Down the Arguments

```python
DataLoaders([], [(s, yb)])
#           │      │
#           │      └── Validation: ONE batch containing (images, labels)
#           └───────── Training: EMPTY (no training data)
```

**Why empty training `[]`?**
- We're NOT training anything
- We just want to run the model on our generated images
- Empty list = "skip training entirely"

**Why `[(s, yb)]` for validation?**
- `s` = our 256 generated images (shape: `[256, 1, 32, 32]`)
- `yb` = labels (we don't actually need these, but DataLoader expects them)
- The outer `[...]` makes it a list with ONE batch
- We're putting our generated images in the "validation" slot so `capture_preds()` can process them

### Visual Representation

```
DataLoaders([], [(s, yb)])

┌─────────────────────────────────────────────────────┐
│ Training DataLoader:  []                            │
│   └── Empty! No batches to iterate over             │
│                                                     │
│ Validation DataLoader: [(s, yb)]                    │
│   └── One batch:                                    │
│       ├── s  = 256 generated images                 │
│       └── yb = labels (placeholder, not used)       │
└─────────────────────────────────────────────────────┘
```

---

## Part 2: Understanding `TrainLearner`

```python
clearn = TrainLearner(
    model,                          # Our feature extractor (classifier with head removed)
    DataLoaders([], [(s, yb)]),     # Data setup explained above
    loss_func=fc.noop,              # No loss calculation needed
    cbs=[DeviceCB()],               # Just move data to GPU
    opt_func=None                   # No optimizer (not training!)
)
```

### What Each Argument Means

| Argument | Value | Why? |
|----------|-------|------|
| `model` | Feature extractor | The classifier with final layers removed - outputs 512-dim features |
| `dls` | `DataLoaders([], [(s,yb)])` | Our generated images as "validation" data |
| `loss_func` | `fc.noop` | "No operation" - we don't need loss, just features |
| `cbs` | `[DeviceCB()]` | Only callback: move tensors to GPU |
| `opt_func` | `None` | No optimizer because we're not training |

### What is `fc.noop`?

```python
# fc.noop is literally a function that does nothing:
def noop(*args, **kwargs):
    pass
```

We use it because `TrainLearner` requires a loss function, but we don't actually need one - we just want to run the forward pass.

---

## Part 3: Understanding `capture_preds()`

```python
feats2, y2 = clearn.capture_preds()
```

### What Is `capture_preds()`?

`capture_preds()` is a **fastai/miniai method** - it's defined on the `Learner` class (or `TrainLearner` in miniai). It's a convenience method that handles all the boilerplate code for running inference.

### What Does `capture_preds()` Do?

It runs the model on the **validation data** and captures:
1. **Model outputs** (predictions/features)
2. **True labels** (ground truth)

### Simplified Version of What It Looks Like Internally

```python
class Learner:
    def capture_preds(self):
        self.model.eval()                      # Set to evaluation mode
        preds, targets = [], []
        
        with torch.no_grad():                  # Disable gradient computation
            for xb, yb in self.dls.valid:      # Loop through validation data
                xb = xb.to(device)             # Move to GPU
                pred = self.model(xb)          # Forward pass
                preds.append(pred.cpu())       # Store prediction
                targets.append(yb)             # Store label
        
        return torch.cat(preds), torch.cat(targets)  # Concatenate all batches
```

### Why Use `capture_preds()` Instead of Manual Code?

It saves you from writing boilerplate:

```python
# ❌ Without capture_preds (manual way - tedious!):
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for xb, yb in validation_loader:
        xb = xb.cuda()
        pred = model(xb)
        all_preds.append(pred.cpu())
        all_labels.append(yb)
feats = torch.cat(all_preds)
labels = torch.cat(all_labels)

# ✅ With capture_preds (clean way - one line!):
feats, labels = learn.capture_preds()
```

**`capture_preds()` handles all the details for you:**
- Sets model to eval mode (`model.eval()`)
- Disables gradients (`torch.no_grad()`)
- Moves data to correct device (GPU/CPU)
- Concatenates results from all batches
- Returns clean tensors ready to use

### Step-by-Step Execution

```
┌─────────────────────────────────────────────────────────────────┐
│ capture_preds() execution:                                      │
│                                                                 │
│ 1. Switch model to eval mode (model.eval())                     │
│                                                                 │
│ 2. Iterate over validation DataLoader:                          │
│    └── Only ONE batch: (s, yb)                                  │
│        ├── s  → moved to GPU                                    │
│        └── yb → stored for return                               │
│                                                                 │
│ 3. Run forward pass:                                            │
│    └── features = model(s)                                      │
│        └── Shape: [256, 512]                                    │
│            (256 images × 512 features each)                     │
│                                                                 │
│ 4. Collect and return:                                          │
│    └── return (all_predictions, all_labels)                     │
│        └── (feats2, y2)                                         │
└─────────────────────────────────────────────────────────────────┘
```

### What Are `feats2` and `y2`?

| Variable | Contains | Shape | Meaning |
|----------|----------|-------|---------|
| `feats2` | Model outputs | `[256, 512]` (before squeeze) | 512-dimensional feature vector for each of 256 generated images |
| `y2` | Labels | `[256]` | The labels we passed in (not actually used for FID) |

---

## Part 4: Why `.squeeze()`?

```python
feats2 = feats2.float().squeeze()
```

### What Does `squeeze()` Do?

It **removes dimensions of size 1** from a tensor.

### Why Do We Need It?

Sometimes the model output has extra dimensions:

```python
# Possible shapes before squeeze:
feats2.shape = torch.Size([256, 512, 1, 1])  # Extra spatial dims
# or
feats2.shape = torch.Size([1, 256, 512])    # Extra batch dim

# After squeeze:
feats2.shape = torch.Size([256, 512])       # Clean 2D tensor!
```

### Visual Example

```
Before squeeze():
┌─────────────────────────────────────┐
│ Shape: [256, 512, 1, 1]             │
│                    ↑  ↑             │
│                    │  └── Size 1!   │
│                    └───── Size 1!   │
└─────────────────────────────────────┘
                    │
                    ▼ squeeze()
┌─────────────────────────────────────┐
│ Shape: [256, 512]                   │
│                                     │
│ Clean 2D matrix:                    │
│   256 images × 512 features         │
└─────────────────────────────────────┘
```

### Why `.float()` Too?

The model might output in mixed precision (float16/bfloat16) for speed. `.float()` converts to float32 for accurate FID/KID calculations.

---

## Complete Flow Summary

```
┌─────────────────────────────────────────────────────────────────┐
│                    FEATURE EXTRACTION FLOW                       │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  INPUT: s (256 generated images)                                │
│         Shape: [256, 1, 32, 32]                                 │
│                    │                                            │
│                    ▼                                            │
│  ┌─────────────────────────────────────────┐                    │
│  │ DataLoaders([], [(s, yb)])              │                    │
│  │   • Training: empty                     │                    │
│  │   • Validation: our generated images    │                    │
│  └─────────────────────────────────────────┘                    │
│                    │                                            │
│                    ▼                                            │
│  ┌─────────────────────────────────────────┐                    │
│  │ clearn.capture_preds()                  │                    │
│  │   • Runs model on validation data       │                    │
│  │   • Returns (outputs, labels)           │                    │
│  └─────────────────────────────────────────┘                    │
│                    │                                            │
│                    ▼                                            │
│  ┌─────────────────────────────────────────┐                    │
│  │ model(s)  →  feats2                     │                    │
│  │   Shape: [256, 512, 1, 1] (maybe)       │                    │
│  └─────────────────────────────────────────┘                    │
│                    │                                            │
│                    ▼                                            │
│  ┌─────────────────────────────────────────┐                    │
│  │ feats2.float().squeeze()                │                    │
│  │   • .float() → convert to float32       │                    │
│  │   • .squeeze() → remove size-1 dims     │                    │
│  │   Shape: [256, 512]                     │                    │
│  └─────────────────────────────────────────┘                    │
│                                                                 │
│  OUTPUT: feats2                                                 │
│          256 images × 512 features                              │
│          Ready for FID/KID comparison with real image features! │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## Key Takeaways

| Question | Answer |
|----------|--------|
| Why empty `[]` for training? | We're not training, just extracting features |
| Why `[(s, yb)]` for validation? | Wraps our generated images as one validation batch |
| Is `capture_preds()` a fastai function? | Yes! It's a method on the `Learner` class in fastai/miniai |
| What does `capture_preds()` do? | Runs forward pass on validation data, returns outputs and labels |
| Why use `capture_preds()`? | Handles eval mode, no_grad, device transfer, and concatenation in one line |
| What is `feats2`? | 512-dimensional features for each generated image |
| What is `y2`? | Labels (not used for FID, just required by DataLoader) |
| Why `squeeze()`? | Removes extra dimensions of size 1 for clean [256, 512] shape |

**What does the output show?**

`torch.Size([256, 512])` - 256 generated images, each with 512 features.

Now we have:
- `feats`: Features from real images (10000 × 512)
- `feats2`: Features from generated images (256 × 512)

Let's compare them!

### Computing Statistics (Mean and Covariance)

In [ ]:
# Compute the mean of real image features
# .mean(0) averages across all images, keeping the feature dimension
means = feats.mean(0)
means.shape

torch.Size([512])

**What does this show?**

`torch.Size([512])` - A 512-dimensional vector representing the "average" feature values across all real images.

```
Mean feature vector:
┌────────────────────────────────────────────┐
│ avg_feat_1, avg_feat_2, ..., avg_feat_512  │
└────────────────────────────────────────────┘
```

In [ ]:
# Compute the covariance matrix of real image features
# .T.cov() transposes then computes covariance
# This gives us the covariance between all pairs of features
covs = feats.T.cov()
covs.shape

**What does this show?**

`torch.Size([512, 512])` - A 512×512 covariance matrix.

**What is a covariance matrix?**

Entry (i, j) tells us how features i and j vary together:
- Positive: When feature i is high, feature j tends to be high
- Negative: When feature i is high, feature j tends to be low
- Zero: Features i and j are independent

```
Covariance Matrix:
┌───────────────────────────────────────┐
│ var(f1)    cov(f1,f2)  ...  cov(f1,f512) │
│ cov(f2,f1) var(f2)     ...  cov(f2,f512) │
│ ...        ...         ...  ...          │
│ cov(f512,f1) ...       ...  var(f512)    │
└───────────────────────────────────────┘
```

The diagonal contains variances; off-diagonal contains covariances.

We're basically looking for two data sets where their covariance matrices are kind of the same and their means are also kind of the same.

### 🎮 Interactive: μ and Σ in real 3D — the only two things FID keeps

`_calc_stats` throws away every individual image and keeps just **μ** (where the feature cloud sits) and **Σ** (how it spreads and tilts). This lab shows both *as 3D objects*: μ is a solid ball at the cloud's center, Σ becomes principal-axis rods and a translucent **2σ ellipsoid** wrapping the points. The orange cloud is fixed (think: real features); you deform the teal one (generated features) and the **exact Fréchet distance** between the two Gaussians updates live.

**What to try:**
- **Drag to orbit, scroll to zoom** — convince yourself the ellipsoid really hugs the points.
- Walk the stages: ① the raw cloud → ② μ appears → ③ Σ's axes grow → ④ the ellipsoid wraps → ⑤ the two-cloud comparison.
- Push the **Δmean / spread / tilt** sliders one at a time and watch which part of the picture (and which part of the FID number) each one moves.
- Hit **✨ match** to morph the teal cloud onto the orange one — the FID readout collapses toward 0.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/gaussian_stats_3d.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/gaussian_stats_3d.html", height="800px")

### Matrix Square Root

FID requires computing the matrix square root of Σ₁Σ₂. This is the trickiest part!

**What is a matrix square root?**

For a matrix A, the square root √A is a matrix B such that B × B = A.

We provide two implementations:
1. Newton-Schulz iteration (GPU-friendly)
2. scipy.linalg.sqrtm (more robust)

In [ ]:
#|export
def _sqrtm_newton_schulz(mat, num_iters=100):
    """
    Compute matrix square root using Newton-Schulz iteration.
    
    This is a GPU-friendly iterative algorithm.
    
    Parameters:
    -----------
    mat : torch.Tensor
        Square positive semi-definite matrix
    num_iters : int
        Maximum iterations
        
    Returns:
    --------
    torch.Tensor
        Matrix square root of mat
    """
    # Normalize the matrix for numerical stability
    mat_nrm = mat.norm()
    mat = mat.double()  # Use double precision
    Y = mat / mat_nrm
    n = len(mat)
    I = torch.eye(n, n).to(mat)
    Z = torch.eye(n, n).to(mat)

    # Newton-Schulz iteration
    for i in range(num_iters):
        T = (3 * I - Z @ Y) / 2
        Y, Z = Y @ T, T @ Z
        res = Y * mat_nrm.sqrt()
        # Check convergence
        if ((mat - (res @ res)).norm() / mat_nrm).abs() <= 1e-6:
            break
    return res

**What does the code above do?**

Implements the Newton-Schulz algorithm for matrix square root:

1. **Normalize** the matrix for stability
2. **Iterate** using the Newton-Schulz update:
   - T = (3I - ZY) / 2
   - Y = YT
   - Z = TZ
3. **Check convergence**: Is (√A)² ≈ A?
4. **Return** the result

This is GPU-friendly because it only uses matrix multiplications.

### 🎮 Interactive: Newton–Schulz — a square root with no `sqrt` anywhere

`_sqrtm_newton_schulz` finds √M using **only matrix multiplies** (GPU-friendly!). This stepper lets you run the iteration one step at a time on a real 2×2 matrix and *see* convergence three ways at once: the **Y matrix** (green cells = digits that have locked in), the check **Y·Y → M**, and an **ellipse view** where Y bends the unit circle until it matches the true √M shape.

**What to try:**
- Pick a matrix: *easy* (nearly diagonal), *a real covariance*, or *stiff* (ill-conditioned) — and compare how many steps each needs.
- Use **Step ⏭** to advance a single iteration; watch `Y·Y` creep toward M cell by cell.
- **Click anywhere on the log-error curve** to jump the whole display to that iteration.
- Follow the 4-phase code panel: normalize → iterate → rescale → check — the exact structure of the notebook's function.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/newton_schulz_stepper.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/newton_schulz_stepper.html", height="780px")

In [ ]:
#|export
def _calc_stats(feats):
    """
    Calculate mean and covariance of features.
    
    Parameters:
    -----------
    feats : torch.Tensor
        Feature matrix of shape (n_samples, n_features)
        
    Returns:
    --------
    tuple: (mean, covariance)
    """
    feats = feats.squeeze()
    return feats.mean(0), feats.T.cov()


def _calc_fid(m1, c1, m2, c2):
    """
    Calculate Fréchet Inception Distance.
    
    FID = ||m1 - m2||² + Tr(c1 + c2 - 2*sqrt(c1*c2))
    
    Parameters:
    -----------
    m1, c1 : mean and covariance of real features
    m2, c2 : mean and covariance of generated features
    
    Returns:
    --------
    float : FID score (lower is better)
    """
    # Compute matrix square root of c1*c2
    # Using scipy for robustness (can use _sqrtm_newton_schulz for GPU)
    csr = tensor(linalg.sqrtm(c1 @ c2, blocksize=256).real)
    
    # FID formula:
    # ||m1 - m2||² + Tr(c1) + Tr(c2) - 2*Tr(sqrt(c1*c2))
    mean_diff_sq = ((m1 - m2) ** 2).sum()  # Squared distance between means
    trace_sum = c1.trace() + c2.trace()    # Sum of variances
    trace_sqrt = 2 * csr.trace()           # Cross term
    
    return (mean_diff_sq + trace_sum - trace_sqrt).item()

**What does the code above do?**

**`_calc_stats`**: Computes mean and covariance of feature vectors.

**`_calc_fid`**: Computes the FID score:

$$FID = \underbrace{||m_1 - m_2||^2}_{\text{mean difference}} + \underbrace{Tr(Σ_1) + Tr(Σ_2)}_{\text{sum of variances}} - \underbrace{2 \cdot Tr(\sqrt{Σ_1 Σ_2})}_{\text{cross term}}$$

**Interpretation:**
- First term: How different are the average features?
- Second term: Total variance in both distributions
- Third term: How similar are the covariance structures?

# Understanding `csr = tensor(linalg.sqrtm(c1 @ c2, blocksize=256).real)`

## The Full Context

```python
def _calc_fid(m1, c1, m2, c2):
    csr = tensor(linalg.sqrtm(c1 @ c2, blocksize=256).real)
    
    mean_diff_sq = ((m1 - m2) ** 2).sum()
    trace_sum = c1.trace() + c2.trace()
    trace_sqrt = 2 * csr.trace()
    
    return (mean_diff_sq + trace_sum - trace_sqrt).item()
```

This line computes the **matrix square root of the product of two covariance matrices**, which is the trickiest part of the FID formula. Let's decompose it from the inside out.

---

## Step-by-Step Decomposition

### Step 1: `c1 @ c2` — Matrix Product of Two Covariances

```python
c1 @ c2
# c1: covariance of real features      → shape [512, 512]
# c2: covariance of generated features  → shape [512, 512]
# Result: their matrix product          → shape [512, 512]
```

This is the **Σ₁Σ₂** term from the FID formula. It captures how the two covariance structures interact.

---

### Step 2: `linalg.sqrtm(...)` — Matrix Square Root

```python
linalg.sqrtm(c1 @ c2, blocksize=256)
```

This computes the **matrix square root** — a matrix `B` such that `B @ B = c1 @ c2`.

This is completely different from element-wise square root. Think of it as the matrix analog of scalar square root:

```
Scalar:  √9 = 3         because  3 × 3 = 9
Matrix:  sqrtm(A) = B   because  B @ B = A
```

#### The `blocksize=256` Parameter

The function signature is:

```python
scipy.linalg.sqrtm(A, disp=True, blocksize=64)
```

`blocksize` controls how the internal **Schur decomposition** algorithm partitions the matrix — larger blocks can be faster but use more memory. It doesn't change the mathematical result, only the computation strategy.

#### A Note on the Original Code

The original notebook actually has a subtle bug:

```python
# What was written (bug):
linalg.sqrtm(c1 @ c2, 256)
#                      ^^^
#                      This goes to `disp`, not `blocksize`!

# What was likely intended:
linalg.sqrtm(c1 @ c2, blocksize=256)
```

Since `256` is positionally mapped to `disp` (not `blocksize`), and any nonzero integer is truthy (behaves like `disp=True`), it accidentally works — `blocksize` silently falls back to its default of `64`.

---

### Step 3: `.real` — Discard Imaginary Noise

```python
linalg.sqrtm(c1 @ c2, blocksize=256).real
```

`sqrtm` returns a **complex-valued** numpy array. In theory, for positive semi-definite matrices the result should be purely real. In practice, floating-point arithmetic introduces tiny imaginary artifacts:

```
Actual result:    42.0 + 1.2e-15j    ← tiny imaginary noise
What we want:     42.0               ← just the real part
```

`.real` strips those negligible imaginary parts.

---

### Step 4: `tensor(...)` — Convert to PyTorch

```python
csr = tensor(linalg.sqrtm(c1 @ c2, blocksize=256).real)
```

`scipy.linalg.sqrtm` returns a **numpy array**. Since the rest of the code uses PyTorch tensors, we wrap it in `tensor(...)` to convert it back to a PyTorch tensor.

The result `csr` (short for **c**ovariance **s**quare **r**oot) is then used for:

```python
trace_sqrt = 2 * csr.trace()  # 2 × Tr(√(Σ₁Σ₂))
```

---

## How It Fits Into the FID Formula

$$FID = \underbrace{||\mu_1 - \mu_2||^2}_{\text{mean\_diff\_sq}} + \underbrace{Tr(\Sigma_1) + Tr(\Sigma_2)}_{\text{trace\_sum}} - \underbrace{2 \cdot Tr(\sqrt{\Sigma_1 \Sigma_2})}_{\text{trace\_sqrt}}$$

| Code Variable | Formula Term | Meaning |
|---------------|-------------|---------|
| `mean_diff_sq` | $\|\|\mu_1 - \mu_2\|\|^2$ | How different are the average features? |
| `trace_sum` | $Tr(\Sigma_1) + Tr(\Sigma_2)$ | Total variance in both distributions |
| `trace_sqrt` | $2 \cdot Tr(\sqrt{\Sigma_1 \Sigma_2})$ | How similar are the covariance structures? |

The `csr` line computes the core of the third term — the one that measures covariance alignment between real and generated feature distributions.

---

## Summary: Reading the Line Left to Right

```
csr = tensor( linalg.sqrtm( c1 @ c2,  blocksize=256 ).real )
 │      │         │           │            │            │
 │      │         │           │            │            └─ Strip imaginary noise
 │      │         │           │            └─ Algorithm tuning (block size)
 │      │         │           └─ Multiply the two covariance matrices
 │      │         └─ Compute matrix square root (scipy)
 │      └─ Convert numpy array → PyTorch tensor
 └─ Store as "covariance square root"
```

In [ ]:
# Calculate statistics for both real and generated features
s1, s2 = _calc_stats(feats), _calc_stats(feats2)

In [ ]:
# Calculate FID!
# Lower is better. FID=0 would mean identical distributions.
_calc_fid(*s1, *s2)

33.834

**What does the result mean?**

An FID of ~34 means there's some difference between real and generated images, but they're reasonably similar.

**FID Reference Values:**
- FID ≈ 0: Generated images are indistinguishable from real
- FID < 10: Excellent quality
- FID 10-50: Good quality
- FID > 100: Poor quality

(Note: These are rough guidelines; actual values depend on the dataset and feature extractor.)

### 🎮 Interactive: the FID formula, dissected term by term

FID = ‖μ₁−μ₂‖² + Tr(Σ₁ + Σ₂ − 2√(Σ₁Σ₂)) — two ideas glued together: **how far apart are the centers?** and **how differently shaped are the clouds?** Here the formula itself is clickable: select either term to isolate what it measures on a live 2-D plane, then feed it with its own slider. The number shown is the *exact* Fréchet formula computed on these clouds.

**What to try:**
- **Click ‖μ₁−μ₂‖² in the formula card** — the plane dims everything except the two centers and the dashed gap between them. Now drag the **Δmean** slider: only the orange bar grows.
- **Click the trace term** — now only shape differences matter. Play with **spread** and **tilt** and watch the teal bar respond while the orange one stays put.
- Stage ④ explains the weird-looking √(Σ₁Σ₂): it's the "shared shape" credit that stops the trace term from double-counting.
- Press **✨ make it perfect** and **💥 make it awful** to see the formula's two extremes, then eyeball where this notebook's FID ≈ 33.8 sits between them.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/fid_formula_anatomy.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/fid_formula_anatomy.html", height="740px")

---

## Section 4: KID (Kernel Inception Distance)

**KID** is an alternative to FID with some advantages:
- No assumption of Gaussian distribution
- Unbiased estimator (FID is biased for small sample sizes)
- Uses Maximum Mean Discrepancy (MMD) with a polynomial kernel

In [ ]:
#|export
def _squared_mmd(x, y):
    """
    Compute squared Maximum Mean Discrepancy with polynomial kernel.
    
    MMD measures the difference between two distributions using a kernel.
    
    Parameters:
    -----------
    x : torch.Tensor
        Features from distribution 1, shape (m, d)
    y : torch.Tensor
        Features from distribution 2, shape (n, d)
        
    Returns:
    --------
    float : Squared MMD
    """
    def k(a, b):
        """
        Polynomial kernel: k(a,b) = (a·b/d + 1)³
        
        This measures similarity between feature vectors.
        """
        # Transpose last two dimensions to compute dot product of all pairs.
        # This ensures compatibility with both 2D and batched tensors.
        return (a @ b.transpose(-2, -1) / a.shape[-1] + 1) ** 3
    
    m, n = x.shape[-2], y.shape[-2]  # Number of samples
    
    # Compute kernel matrices
    kxx = k(x, x)  # Kernel between x samples (m x m)
    kyy = k(y, y)  # Kernel between y samples (n x n)
    kxy = k(x, y)  # Kernel between x and y (m x n)
    
    # MMD formula (unbiased estimator)
    # We subtract diagonal elements because k(xi, xi) shouldn't count
    kxx_sum = kxx.sum([-1, -2]) - kxx.diagonal(0, -1, -2).sum(-1)
    kyy_sum = kyy.sum([-1, -2]) - kyy.diagonal(0, -1, -2).sum(-1)
    kxy_sum = kxy.sum([-1, -2])
    
    return kxx_sum / m / (m - 1) + kyy_sum / n / (n - 1) - kxy_sum * 2 / m / n

**What does the code above do?**

Computes **Maximum Mean Discrepancy (MMD)** - a way to measure distribution difference.

**The Kernel:**
$$k(a, b) = \left(\frac{a \cdot b}{d} + 1\right)^3$$

Where:
- $a, b$: Feature vectors (e.g., activations from a pre-trained model like Inception).
- $d$: The dimensionality of the feature vectors.

This polynomial kernel measures similarity between feature vectors.

**The MMD Formula:**
$$MMD^2 = \frac{1}{m(m-1)}\sum_{i \neq j} k(x_i, x_j) + \frac{1}{n(n-1)}\sum_{i \neq j} k(y_i, y_j) - \frac{2}{mn}\sum_{i,j} k(x_i, y_j)$$

Where:
- $x$: Feature vectors of **real images** (sample size $m$).
- $y$: Feature vectors of **generated images** (sample size $n$).

**Intuition:**
- First term: Average similarity within **real images** ($x$)
- Second term: Average similarity within **generated images** ($y$)
- Third term: Average similarity between **real and generated** images
- If distributions match, these should be similar → MMD ≈ 0

# Understanding `_squared_mmd` - A Deep Dive

## The Code in Question

```python
#|export
def _squared_mmd(x, y):
    """
    Compute squared Maximum Mean Discrepancy with polynomial kernel.
    
    MMD measures the difference between two distributions using a kernel.
    
    Parameters:
    -----------
    x : torch.Tensor
        Features from distribution 1, shape (m, d)
    y : torch.Tensor
        Features from distribution 2, shape (n, d)
        
    Returns:
    --------
    float : Squared MMD
    """
    def k(a, b):
        """
        Polynomial kernel: k(a,b) = (a·b/d + 1)³
        
        This measures similarity between feature vectors.
        """
        # Transpose last two dimensions to compute dot product of all pairs.
        # This ensures compatibility with both 2D and batched tensors.
        return (a @ b.transpose(-2, -1) / a.shape[-1] + 1) ** 3
    
    m, n = x.shape[-2], y.shape[-2]  # Number of samples
    
    # Compute kernel matrices
    kxx = k(x, x)  # Kernel between x samples (m x m)
    kyy = k(y, y)  # Kernel between y samples (n x n)
    kxy = k(x, y)  # Kernel between x and y (m x n)
    
    # MMD formula (unbiased estimator)
    # We subtract diagonal elements because k(xi, xi) shouldn't count
    kxx_sum = kxx.sum([-1, -2]) - kxx.diagonal(0, -1, -2).sum(-1)
    kyy_sum = kyy.sum([-1, -2]) - kyy.diagonal(0, -1, -2).sum(-1)
    kxy_sum = kxy.sum([-1, -2])
    
    return kxx_sum / m / (m - 1) + kyy_sum / n / (n - 1) - kxy_sum * 2 / m / n
```

---

## Part 1: The Polynomial Kernel Function

```python
return (a @ b.transpose(-2, -1) / a.shape[-1] + 1) ** 3
```

---

## Understanding the Transpose: `b.transpose(-2, -1)`

### The Setup

Let's say we have:
- `a` = features from set A, shape `[m, d]` (m samples, d features each)
- `b` = features from set B, shape `[n, d]` (n samples, d features each)

For our FID/KID context:
- `m` = number of images (e.g., 256)
- `d` = number of features (e.g., 512)

```
a shape: [256, 512]  →  256 images, each with 512 features
b shape: [256, 512]  →  256 images, each with 512 features
```

### Why Transpose?

We want to compute the **dot product between every pair of samples**. 

```
┌─────────────────────────────────────────────────────────────────┐
│  GOAL: Compute similarity between ALL pairs of samples          │
│                                                                 │
│  a has m samples: [a₁, a₂, ..., aₘ]  (each is 512-dim vector)  │
│  b has n samples: [b₁, b₂, ..., bₙ]  (each is 512-dim vector)  │
│                                                                 │
│  We want a matrix where entry (i,j) = aᵢ · bⱼ (dot product)    │
└─────────────────────────────────────────────────────────────────┘
```

### Matrix Multiplication Review

For matrix multiplication `A @ B` to work:
- A shape: `[m, k]`
- B shape: `[k, n]`
- Result: `[m, n]`

**The inner dimensions must match!**

### Without Transpose (WRONG)

```python
a @ b  # [256, 512] @ [256, 512] → ERROR! 512 ≠ 256
```

### With Transpose (CORRECT)

```python
b.transpose(-2, -1)  # [256, 512] → [512, 256]

a @ b.transpose(-2, -1)  # [256, 512] @ [512, 256] → [256, 256] ✓
```

### Visual Explanation

```
┌─────────────────────────────────────────────────────────────────┐
│                     MATRIX MULTIPLICATION                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   a: [m, d]              b.T: [d, n]           result: [m, n]   │
│                                                                 │
│   ┌─────────┐            ┌─────────┐           ┌─────────┐      │
│   │ a₁ ───→ │            │ ↓  ↓  ↓ │           │         │      │
│   │ a₂ ───→ │     @      │ b₁ b₂ bₙ│     =     │  aᵢ·bⱼ  │      │
│   │ ...     │            │         │           │         │      │
│   │ aₘ ───→ │            │         │           │         │      │
│   └─────────┘            └─────────┘           └─────────┘      │
│   [256, 512]             [512, 256]            [256, 256]       │
│                                                                 │
│   Each row of a          Each column of b.T    Entry (i,j) =    │
│   is one sample          is one sample         dot(aᵢ, bⱼ)      │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### What `-2, -1` Means

```python
b.transpose(-2, -1)
```

- `-1` = last dimension (features dimension, d=512)
- `-2` = second-to-last dimension (samples dimension, m=256)

This swaps the last two dimensions, which is equivalent to `.T` for 2D tensors but also works for batched (3D+) tensors.

---

## The Complete Kernel Formula

```python
(a @ b.transpose(-2, -1) / a.shape[-1] + 1) ** 3
```

Let's trace through each step:

| Step | Code | Shape | What it does |
|------|------|-------|--------------|
| 1 | `b.transpose(-2, -1)` | [512, 256] | Transpose b |
| 2 | `a @ b.transpose(-2, -1)` | [256, 256] | All pairwise dot products |
| 3 | `/ a.shape[-1]` | [256, 256] | Divide by d (512) to normalize |
| 4 | `+ 1` | [256, 256] | Shift (ensures positive values) |
| 5 | `** 3` | [256, 256] | Cube (polynomial kernel degree 3) |

### Why Each Step?

| Step | Why? |
|------|------|
| Transpose | Align dimensions for matrix multiplication |
| Matrix multiply | Compute all pairwise dot products efficiently |
| Divide by d | Normalize so values don't explode with high dimensions |
| Add 1 | Ensure values are positive before cubing; also adds a "bias" term |
| Cube | Polynomial kernel of degree 3 (captures non-linear relationships) |

### The Result: A Kernel Matrix

```
┌─────────────────────────────────────────────────────────────────┐
│                    KERNEL MATRIX k(a, b)                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│         b₁      b₂      b₃    ...    bₙ                        │
│       ┌──────┬──────┬──────┬──────┬──────┐                      │
│   a₁  │k(a₁,b₁)│k(a₁,b₂)│k(a₁,b₃)│ ... │k(a₁,bₙ)│              │
│       ├──────┼──────┼──────┼──────┼──────┤                      │
│   a₂  │k(a₂,b₁)│k(a₂,b₂)│k(a₂,b₃)│ ... │k(a₂,bₙ)│              │
│       ├──────┼──────┼──────┼──────┼──────┤                      │
│   a₃  │k(a₃,b₁)│k(a₃,b₂)│k(a₃,b₃)│ ... │k(a₃,bₙ)│              │
│       ├──────┼──────┼──────┼──────┼──────┤                      │
│  ...  │ ...  │ ...  │ ...  │ ... │ ...  │                       │
│       ├──────┼──────┼──────┼──────┼──────┤                      │
│   aₘ  │k(aₘ,b₁)│k(aₘ,b₂)│k(aₘ,b₃)│ ... │k(aₘ,bₙ)│              │
│       └──────┴──────┴──────┴──────┴──────┘                      │
│                                                                 │
│   Shape: [m, n] = [256, 256]                                    │
│   Each entry k(aᵢ,bⱼ) = ((aᵢ·bⱼ)/d + 1)³                       │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## Part 2: The Diagonal Subtraction

```python
kxx_sum = kxx.sum([-1, -2]) - kxx.diagonal(0, -1, -2).sum(-1)
kyy_sum = kyy.sum([-1, -2]) - kyy.diagonal(0, -1, -2).sum(-1)
kxy_sum = kxy.sum([-1, -2])
```

### Why Subtract the Diagonal?

The MMD formula requires summing over **pairs where i ≠ j**:

$$\sum_{i \neq j} k(x_i, x_j)$$

This means we should **NOT** include `k(x₁, x₁)`, `k(x₂, x₂)`, etc. (self-comparisons).

### The Problem with `kxx`

When we compute `k(x, x)`, the diagonal contains self-similarities:

```
┌─────────────────────────────────────────────────────────────────┐
│                         kxx = k(x, x)                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│         x₁      x₂      x₃    ...    xₘ                        │
│       ┌──────┬──────┬──────┬──────┬──────┐                      │
│   x₁  │██████│k(x₁,x₂)│k(x₁,x₃)│ ... │k(x₁,xₘ)│                │
│       ├──────┼──────┼──────┼──────┼──────┤                      │
│   x₂  │k(x₂,x₁)│██████│k(x₂,x₃)│ ... │k(x₂,xₘ)│                │
│       ├──────┼──────┼──────┼──────┼──────┤                      │
│   x₃  │k(x₃,x₁)│k(x₃,x₂)│██████│ ... │k(x₃,xₘ)│                │
│       ├──────┼──────┼──────┼──────┼──────┤                      │
│  ...  │ ...  │ ...  │ ...  │██████│ ...  │                      │
│       ├──────┼──────┼──────┼──────┼──────┤                      │
│   xₘ  │k(xₘ,x₁)│k(xₘ,x₂)│k(xₘ,x₃)│ ... │██████│                │
│       └──────┴──────┴──────┴──────┴──────┘                      │
│                                                                 │
│   ██████ = DIAGONAL = k(xᵢ, xᵢ) = self-similarity              │
│                                                                 │
│   We want to SUM everything EXCEPT the diagonal!                │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### The Solution

```python
kxx.sum([-1, -2])                  # Sum ALL elements (including diagonal)
- kxx.diagonal(0, -1, -2).sum(-1)  # Subtract diagonal elements
= kxx_sum                          # Sum of off-diagonal elements only!
```

### Breaking Down `kxx.diagonal(0, -1, -2)`

```python
kxx.diagonal(0, -1, -2)
#           │   │   │
#           │   │   └── dim2: last dimension
#           │   └────── dim1: second-to-last dimension  
#           └────────── offset: 0 (main diagonal)
```

This extracts the main diagonal of the matrix formed by the last two dimensions.

```
kxx shape: [256, 256]
kxx.diagonal(0, -1, -2) shape: [256]  ← The 256 diagonal elements
```

### Step-by-Step Example

```python
# Example with small 3x3 matrix
kxx = tensor([
    [k(x₁,x₁), k(x₁,x₂), k(x₁,x₃)],
    [k(x₂,x₁), k(x₂,x₂), k(x₂,x₃)],
    [k(x₃,x₁), k(x₃,x₂), k(x₃,x₃)]
])

kxx.sum([-1, -2])  
# = k(x₁,x₁) + k(x₁,x₂) + k(x₁,x₃) + k(x₂,x₁) + k(x₂,x₂) + k(x₂,x₃) + k(x₃,x₁) + k(x₃,x₂) + k(x₃,x₃)
# = sum of ALL 9 elements

kxx.diagonal(0, -1, -2)
# = [k(x₁,x₁), k(x₂,x₂), k(x₃,x₃)]  ← just the 3 diagonal elements

kxx.diagonal(0, -1, -2).sum(-1)
# = k(x₁,x₁) + k(x₂,x₂) + k(x₃,x₃)  ← sum of diagonal

kxx_sum = kxx.sum([-1, -2]) - kxx.diagonal(0, -1, -2).sum(-1)
# = sum of all 9 elements - sum of 3 diagonal elements
# = sum of 6 off-diagonal elements  ✓
```

After .diagonal(), we have a 1D tensor with shape `[256].

For a 1D tensor:

- `-1` refers to the only dimension (which is also 0)
- `.sum(-1)` sums along that dimension, giving a scalar

---

## Why No Diagonal Subtraction for `kxy`?

```python
kxy_sum = kxy.sum([-1, -2])  # No diagonal subtraction!
```

`kxy = k(x, y)` compares samples from **different sets** (real vs generated). There's no "self-comparison" issue because `xᵢ` and `yⱼ` are always different samples!

```
┌─────────────────────────────────────────────────────────────────┐
│                         kxy = k(x, y)                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│         y₁      y₂      y₃    ...    yₙ                        │
│       ┌──────┬──────┬──────┬──────┬──────┐                      │
│   x₁  │k(x₁,y₁)│k(x₁,y₂)│k(x₁,y₃)│ ... │k(x₁,yₙ)│              │
│       ├──────┼──────┼──────┼──────┼──────┤                      │
│   x₂  │k(x₂,y₁)│k(x₂,y₂)│k(x₂,y₃)│ ... │k(x₂,yₙ)│              │
│       ├──────┼──────┼──────┼──────┼──────┤                      │
│   x₃  │k(x₃,y₁)│k(x₃,y₂)│k(x₃,y₃)│ ... │k(x₃,yₙ)│              │
│       └──────┴──────┴──────┴──────┴──────┘                      │
│                                                                 │
│   ALL entries are valid comparisons (x and y are different!)    │
│   No diagonal to subtract - every xᵢ is different from every yⱼ │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## The Final MMD Formula

```python
return kxx_sum / m / (m - 1) + kyy_sum / n / (n - 1) - kxy_sum * 2 / m / n
```

This computes:

$$MMD^2 = \underbrace{\frac{1}{m(m-1)}\sum_{i \neq j} k(x_i, x_j)}_{\text{avg similarity within real}} + \underbrace{\frac{1}{n(n-1)}\sum_{i \neq j} k(y_i, y_j)}_{\text{avg similarity within generated}} - \underbrace{\frac{2}{mn}\sum_{i,j} k(x_i, y_j)}_{\text{avg similarity between real \& generated}}$$

### Breaking Down Each Term

| Term | Code | What it Measures |
|------|------|------------------|
| First | `kxx_sum / m / (m - 1)` | Average similarity within real images |
| Second | `kyy_sum / n / (n - 1)` | Average similarity within generated images |
| Third | `kxy_sum * 2 / m / n` | Average similarity between real & generated |

### Why Different Denominators?

| Denominator | Why? |
|-------------|------|
| `m(m-1)` | Number of pairs where i ≠ j from m samples (excluding self-pairs) |
| `n(n-1)` | Number of pairs where i ≠ j from n samples (excluding self-pairs) |
| `mn` | Number of ALL pairs between m and n samples (no exclusion needed) |

### Intuition

- If real and generated images come from the **same distribution**:
  - Similarity within real ≈ Similarity within generated ≈ Similarity between them
  - **MMD ≈ 0**

- If distributions are **different**:
  - Similarity within each set is high
  - Similarity between sets is lower
  - **MMD > 0**

---

## Complete Flow Diagram

```
┌─────────────────────────────────────────────────────────────────┐
│                    MMD CALCULATION FLOW                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  INPUT:                                                         │
│    x: [m, d] = real image features (e.g., [256, 512])          │
│    y: [n, d] = generated image features (e.g., [256, 512])     │
│                                                                 │
│  STEP 1: Compute Kernel Matrices                                │
│  ─────────────────────────────────                              │
│    kxx = k(x, x)  →  [m, m] = [256, 256]                       │
│    kyy = k(y, y)  →  [n, n] = [256, 256]                       │
│    kxy = k(x, y)  →  [m, n] = [256, 256]                       │
│                                                                 │
│  STEP 2: Sum Off-Diagonal Elements                              │
│  ─────────────────────────────────                              │
│    kxx_sum = sum(kxx) - sum(diagonal(kxx))                     │
│    kyy_sum = sum(kyy) - sum(diagonal(kyy))                     │
│    kxy_sum = sum(kxy)  ← no diagonal subtraction               │
│                                                                 │
│  STEP 3: Compute MMD²                                           │
│  ────────────────────                                           │
│    MMD² = kxx_sum/(m(m-1)) + kyy_sum/(n(n-1)) - 2*kxy_sum/(mn) │
│                                                                 │
│  OUTPUT: MMD² (lower = more similar distributions)              │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## Summary Table

| Question | Answer |
|----------|--------|
| Why `transpose(-2, -1)`? | To align dimensions for matrix multiplication so we get all pairwise dot products |
| What does `a @ b.T` produce? | A matrix where entry (i,j) is the dot product of sample i from a and sample j from b |
| Why `/ a.shape[-1]`? | Normalize by feature dimension (512) to keep values reasonable |
| Why `+ 1`? | Ensure positive values before cubing; adds bias term |
| Why `** 3`? | Polynomial kernel of degree 3 (standard choice for KID) |
| Why subtract diagonal for `kxx`? | Exclude self-comparisons k(xᵢ, xᵢ) for unbiased estimate |
| Why subtract diagonal for `kyy`? | Same reason - exclude self-comparisons k(yᵢ, yᵢ) |
| Why NO diagonal subtraction for `kxy`? | No self-comparisons exist (comparing different sets x vs y) |
| What does `diagonal(0, -1, -2)` do? | Extracts main diagonal from matrix formed by last two dimensions |
| Why `m(m-1)` denominator? | Number of valid pairs excluding self-pairs |
| What does MMD ≈ 0 mean? | Distributions are similar (good generated images!) |
| What does MMD > 0 mean? | Distributions are different |

# Understanding `kyy.diagonal(0, -1, -2)`

## The Function Signature

```python
torch.diagonal(input, offset=0, dim1=0, dim2=1)
```

So when we call:

```python
kyy.diagonal(0, -1, -2)
#            │   │   │
#            │   │   └── dim2 = -2 (second-to-last dimension)
#            │   └────── dim1 = -1 (last dimension)
#            └────────── offset = 0 (main diagonal)
```

---

## What Each Parameter Does

### Parameter 1: `offset = 0`

The **offset** determines WHICH diagonal to extract:

```
┌─────────────────────────────────────────────────────────────────┐
│                    OFFSET EXAMPLES                               │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   offset = 0 (MAIN diagonal)                                    │
│   ┌───┬───┬───┬───┐                                            │
│   │ ■ │   │   │   │                                            │
│   ├───┼───┼───┼───┤                                            │
│   │   │ ■ │   │   │                                            │
│   ├───┼───┼───┼───┤                                            │
│   │   │   │ ■ │   │                                            │
│   ├───┼───┼───┼───┤                                            │
│   │   │   │   │ ■ │                                            │
│   └───┴───┴───┴───┘                                            │
│                                                                 │
│   offset = 1 (one ABOVE main)      offset = -1 (one BELOW main)│
│   ┌───┬───┬───┬───┐                ┌───┬───┬───┬───┐           │
│   │   │ ■ │   │   │                │   │   │   │   │           │
│   ├───┼───┼───┼───┤                ├───┼───┼───┼───┤           │
│   │   │   │ ■ │   │                │ ■ │   │   │   │           │
│   ├───┼───┼───┼───┤                ├───┼───┼───┼───┤           │
│   │   │   │   │ ■ │                │   │ ■ │   │   │           │
│   ├───┼───┼───┼───┤                ├───┼───┼───┼───┤           │
│   │   │   │   │   │                │   │   │ ■ │   │           │
│   └───┴───┴───┴───┘                └───┴───┴───┴───┘           │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

**We use `offset=0`** because we want the **main diagonal** (self-comparisons like k(y₁,y₁), k(y₂,y₂), etc.)

---

### Parameters 2 & 3: `dim1 = -1` and `dim2 = -2`

These specify **which two dimensions form the matrix** from which to extract the diagonal.

```python
dim1 = -1   # Last dimension (columns)
dim2 = -2   # Second-to-last dimension (rows)
```

### For a 2D Tensor (Simple Case)

```python
kyy.shape = [256, 256]
#             │    │
#             │    └── dim -1 (last) = columns
#             └─────── dim -2 (second-to-last) = rows
```

```
┌─────────────────────────────────────────────────────────────────┐
│                    2D TENSOR: kyy [256, 256]                     │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│                    dim -1 (columns)                             │
│                    ─────────────────►                           │
│                    0    1    2   ...  255                       │
│                  ┌────┬────┬────┬────┬────┐                     │
│   dim -2    0    │ D₀ │    │    │    │    │                     │
│   (rows)         ├────┼────┼────┼────┼────┤                     │
│     │       1    │    │ D₁ │    │    │    │                     │
│     │            ├────┼────┼────┼────┼────┤                     │
│     ▼       2    │    │    │ D₂ │    │    │                     │
│                  ├────┼────┼────┼────┼────┤                     │
│            ...   │    │    │    │... │    │                     │
│                  ├────┼────┼────┼────┼────┤                     │
│            255   │    │    │    │    │D₂₅₅│                     │
│                  └────┴────┴────┴────┴────┘                     │
│                                                                 │
│   diagonal(0, -1, -2) extracts: [D₀, D₁, D₂, ..., D₂₅₅]        │
│   Result shape: [256]                                           │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

### For a 3D Tensor (Batched Case)

This is where `dim1` and `dim2` become important!

```python
kyy.shape = [batch, 256, 256]
#              │     │    │
#              │     │    └── dim -1 (last)
#              │     └─────── dim -2 (second-to-last)
#              └───────────── dim -3 (batch dimension)
```

```
┌─────────────────────────────────────────────────────────────────┐
│                    3D TENSOR: kyy [batch, 256, 256]              │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   We have multiple 256×256 matrices stacked:                    │
│                                                                 │
│   Batch 0:              Batch 1:              Batch 2:          │
│   ┌────────────┐        ┌────────────┐        ┌────────────┐    │
│   │ ■          │        │ ■          │        │ ■          │    │
│   │   ■        │        │   ■        │        │   ■        │    │
│   │     ■      │        │     ■      │        │     ■      │    │
│   │       ■    │        │       ■    │        │       ■    │    │
│   └────────────┘        └────────────┘        └────────────┘    │
│                                                                 │
│   dim1=-1, dim2=-2 tells PyTorch:                              │
│   "Extract diagonal from the matrix formed by last 2 dims"      │
│   "Do this for EACH batch independently"                        │
│                                                                 │
│   Result shape: [batch, 256]                                    │
│   (one diagonal per batch)                                      │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## Why Use Negative Indices (-1, -2)?

Negative indices count from the **end**, making the code work for both:

| Tensor Shape | dim -2 | dim -1 | Works? |
|--------------|--------|--------|--------|
| `[256, 256]` | 0 (rows) | 1 (cols) | ✅ |
| `[batch, 256, 256]` | 1 (rows) | 2 (cols) | ✅ |
| `[a, b, 256, 256]` | 2 (rows) | 3 (cols) | ✅ |

If we used positive indices like `diagonal(0, 0, 1)`, it would break for batched tensors!

---

## Complete Example

```python
import torch

# Create a 4x4 matrix
kyy = torch.tensor([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
])

print(kyy.diagonal(0, -1, -2))   # tensor([1, 6, 11, 16]) - main diagonal
print(kyy.diagonal(1, -1, -2))   # tensor([2, 7, 12]) - one above main
print(kyy.diagonal(-1, -1, -2))  # tensor([5, 10, 15]) - one below main
```

### Output Explained

```
Original Matrix:
┌────┬────┬────┬────┐
│  1 │  2 │  3 │  4 │
├────┼────┼────┼────┤
│  5 │  6 │  7 │  8 │
├────┼────┼────┼────┤
│  9 │ 10 │ 11 │ 12 │
├────┼────┼────┼────┤
│ 13 │ 14 │ 15 │ 16 │
└────┴────┴────┴────┘

diagonal(0, -1, -2)  → [1, 6, 11, 16]   (main diagonal: ↘)
diagonal(1, -1, -2)  → [2, 7, 12]       (above main: one step right)
diagonal(-1, -1, -2) → [5, 10, 15]      (below main: one step down)
```

---

## Summary

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `offset` | `0` | Main diagonal (where row index = column index) |
| `dim1` | `-1` | Last dimension (columns of the matrix) |
| `dim2` | `-2` | Second-to-last dimension (rows of the matrix) |

| Question | Answer |
|----------|--------|
| Why `offset=0`? | We want the MAIN diagonal (self-comparisons) |
| Why `dim1=-1`? | Specifies columns; negative index works for any number of batch dimensions |
| Why `dim2=-2`? | Specifies rows; together with dim1, defines which 2D slice to extract diagonal from |
| Why negative indices? | They count from the end, making code work for both batched and non-batched tensors |

---

## In Context of MMD/KID

In the `_squared_mmd` function:

```python
kyy = k(y, y)  # Shape: [256, 256] - kernel matrix comparing y to itself

# We need to sum all elements EXCEPT the diagonal (self-comparisons)
kyy_sum = kyy.sum([-1, -2]) - kyy.diagonal(0, -1, -2).sum(-1)
#         └─── sum all ───┘   └─── subtract diagonal sum ───┘
```

The diagonal contains `k(y₁,y₁), k(y₂,y₂), ...` which are **self-comparisons**. For an unbiased MMD estimate, we need to exclude these, hence we subtract them from the total sum.

# Understanding `kxx.sum([-1, -2])` and `kxx.diagonal(0, -1, -2).sum(-1)`

## The Line We Want to Understand

```python
kxx_sum = kxx.sum([-1, -2]) - kxx.diagonal(0, -1, -2).sum(-1)
```

This computes: **sum of all elements minus sum of diagonal elements = sum of off-diagonal elements only**.

Let's build up to this step by step using a concrete small example.

---

## Setup: A Concrete 3×3 Kernel Matrix

Suppose we have 3 images with features `x₁, x₂, x₃`. The kernel matrix `kxx = k(x, x)` looks like:

```python
kxx = tensor([
    [8., 3., 5.],    # k(x₁,x₁)  k(x₁,x₂)  k(x₁,x₃)
    [3., 7., 2.],    # k(x₂,x₁)  k(x₂,x₂)  k(x₂,x₃)
    [5., 2., 9.]     # k(x₃,x₁)  k(x₃,x₂)  k(x₃,x₃)
])

# Shape: [3, 3]
```

The diagonal elements `[8, 7, 9]` are **self-comparisons** — how similar each image is to *itself*. For the unbiased MMD estimator, we need to **exclude** these.

---

## Part 1: What Does `kxx.sum([-1, -2])` Do?

### First: What Do Negative Dimension Indices Mean?

For a 2D tensor with shape `[3, 3]`:

```
Dimension 0 = Dimension -2 = ROWS     (first axis)
Dimension 1 = Dimension -1 = COLUMNS  (second axis)
```

Negative indices count **from the end**:

```
                    Positive    Negative
                    index       index
                    ─────       ────────
Rows (first):        0           -2
Columns (second):    1           -1
```

### What `.sum([-1, -2])` Means

`kxx.sum([-1, -2])` means: **sum along dimension -1 AND dimension -2**. Since those are the only two dimensions of our 2D matrix, this sums **every single element**:

```
kxx = tensor([
    [8., 3., 5.],
    [3., 7., 2.],
    [5., 2., 9.]
])

kxx.sum([-1, -2])
= 8 + 3 + 5 + 3 + 7 + 2 + 5 + 2 + 9
= 44.
```

It's equivalent to `kxx.sum()` for a 2D tensor. The reason the code specifies `[-1, -2]` instead of just calling `.sum()` is to **support batched tensors** (3D or higher), where you only want to sum over the last two dimensions (the matrix), not the batch dimension.

### Visualizing the Sum

```
┌─────┬─────┬─────┐
│  8  │  3  │  5  │  ─┐
├─────┼─────┼─────┤   │
│  3  │  7  │  2  │   ├── Sum ALL of these = 44
├─────┼─────┼─────┤   │
│  5  │  2  │  9  │  ─┘
└─────┴─────┴─────┘
```

---

## Part 2: What Does `kxx.diagonal(0, -1, -2)` Do?

### The Function Signature

```python
torch.Tensor.diagonal(offset, dim1, dim2)
```

| Parameter | What It Does |
|-----------|-------------|
| `offset`  | Which diagonal: `0` = main, `1` = one above, `-1` = one below |
| `dim1`    | First dimension that forms the 2D matrix |
| `dim2`    | Second dimension that forms the 2D matrix |

### Applied to Our Example

```python
kxx.diagonal(0, -1, -2)
#            │   │   │
#            │   │   └── dim2 = -2 → rows
#            │   └────── dim1 = -1 → columns
#            └────────── offset = 0 → main diagonal
```

This says: "From the 2D matrix formed by dimension -1 (columns) and dimension -2 (rows), extract the main diagonal (offset 0)."

### The Result

```
kxx = tensor([
    [8., 3., 5.],
    [3., 7., 2.],
    [5., 2., 9.]
])

kxx.diagonal(0, -1, -2)
= tensor([8., 7., 9.])     # shape: [3]
```

Visually:

```
┌─────┬─────┬─────┐
│ [8] │     │     │
├─────┼─────┼─────┤
│     │ [7] │     │    diagonal → [8, 7, 9]
├─────┼─────┼─────┤
│     │     │ [9] │
└─────┴─────┴─────┘
```

### Why Negative Indices for dim1 and dim2?

Using `-1` and `-2` (instead of `0` and `1`) makes the code **dimension-agnostic** — it works regardless of how many batch dimensions are prepended:

| Tensor Shape | dim -2 | dim -1 | Extracts diagonal from |
|---|---|---|---|
| `[3, 3]` | dim 0 (rows) | dim 1 (cols) | The single 3×3 matrix |
| `[batch, 3, 3]` | dim 1 (rows) | dim 2 (cols) | Each 3×3 matrix in the batch |
| `[a, b, 3, 3]` | dim 2 (rows) | dim 3 (cols) | Each 3×3 matrix independently |

If we had written `kxx.diagonal(0, 0, 1)`, it would break for batched (3D) tensors.

---

## Part 3: What Does `.sum(-1)` Do After `.diagonal()`?

```python
kxx.diagonal(0, -1, -2).sum(-1)
```

After `.diagonal()`, we have a 1D tensor:

```python
tensor([8., 7., 9.])   # shape: [3]
```

`.sum(-1)` sums along the last (and only) dimension:

```python
tensor([8., 7., 9.]).sum(-1) = 8 + 7 + 9 = 24.
```

Again, `-1` is used instead of `0` for batch compatibility. If the diagonal result were `[batch, 3]`, then `.sum(-1)` would sum along the last dimension, giving one scalar per batch element.

---

## Part 4: Putting It All Together

```python
kxx_sum = kxx.sum([-1, -2]) - kxx.diagonal(0, -1, -2).sum(-1)
```

Step by step:

```
Step 1: kxx.sum([-1, -2])
        = 8 + 3 + 5 + 3 + 7 + 2 + 5 + 2 + 9
        = 44                                      ← sum of ALL 9 elements

Step 2: kxx.diagonal(0, -1, -2)
        = [8, 7, 9]                                ← the 3 diagonal elements

Step 3: kxx.diagonal(0, -1, -2).sum(-1)
        = 8 + 7 + 9
        = 24                                       ← sum of diagonal

Step 4: kxx.sum([-1, -2]) - kxx.diagonal(0, -1, -2).sum(-1)
        = 44 - 24
        = 20                                       ← sum of OFF-diagonal only
```

### Visual Verification

```
┌─────┬─────┬─────┐
│  8  │  3  │  5  │
├─────┼─────┼─────┤       Sum of all 9 elements = 44
│  3  │  7  │  2  │
├─────┼─────┼─────┤       Minus diagonal [8, 7, 9] = 24
│  5  │  2  │  9  │
└─────┴─────┴─────┘       = Off-diagonal sum = 20

Check: 3 + 5 + 3 + 2 + 5 + 2 = 20  ✓
```

The off-diagonal elements are exactly the **pairwise comparisons between different images** — which is what the unbiased MMD estimator requires.

---

## Part 5: Why Does `kxy` NOT Need Diagonal Subtraction?

```python
kxx_sum = kxx.sum([-1, -2]) - kxx.diagonal(0, -1, -2).sum(-1)   # subtract diagonal
kyy_sum = kyy.sum([-1, -2]) - kyy.diagonal(0, -1, -2).sum(-1)   # subtract diagonal
kxy_sum = kxy.sum([-1, -2])                                       # NO subtraction!
```

`kxx = k(x, x)` compares x to itself → diagonal has self-comparisons `k(xᵢ, xᵢ)` → **must exclude**.

`kxy = k(x, y)` compares x to y → entry `(i, i)` is `k(xᵢ, yᵢ)`, which is a comparison between **two different images** from **two different sets** → **no need to exclude anything**.

```
kxx (self-comparison):              kxy (cross-comparison):
┌─────┬─────┬─────┐                ┌─────┬─────┬─────┐
│SELF │  ✓  │  ✓  │                │  ✓  │  ✓  │  ✓  │
├─────┼─────┼─────┤                ├─────┼─────┼─────┤
│  ✓  │SELF │  ✓  │                │  ✓  │  ✓  │  ✓  │
├─────┼─────┼─────┤                ├─────┼─────┼─────┤
│  ✓  │  ✓  │SELF │                │  ✓  │  ✓  │  ✓  │
└─────┴─────┴─────┘                └─────┴─────┴─────┘
 ↑ Must remove SELF                 ↑ ALL entries are valid
```

---

## Summary: The Complete Operation

| Expression | What It Computes | Result for Our Example |
|---|---|---|
| `kxx.sum([-1, -2])` | Sum of **all** elements | 44 |
| `kxx.diagonal(0, -1, -2)` | The diagonal elements | `[8, 7, 9]` |
| `kxx.diagonal(0, -1, -2).sum(-1)` | Sum of diagonal | 24 |
| `kxx.sum([-1, -2]) - kxx.diagonal(0, -1, -2).sum(-1)` | Sum of **off-diagonal** only | 20 |

| Parameter | Value | Role |
|---|---|---|
| `-1` (in `.sum` and `.diagonal`) | Last dimension | Columns of the matrix |
| `-2` (in `.sum` and `.diagonal`) | Second-to-last dimension | Rows of the matrix |
| `0` (in `.diagonal`) | Offset | Main diagonal (where row = column) |

The negative indices exist so the code works for both **2D tensors** `[m, m]` and **batched 3D tensors** `[batch, m, m]` without any changes.

In [ ]:
#|export
def _calc_kid(x, y, maxs=50):
    """
    Calculate Kernel Inception Distance.
    
    Computes MMD in chunks for memory efficiency.
    
    Parameters:
    -----------
    x : torch.Tensor
        Real image features
    y : torch.Tensor
        Generated image features
    maxs : int
        Maximum samples per chunk
        
    Returns:
    --------
    float : KID score (lower is better)
    """
    xs, ys = x.shape[0], y.shape[0]
    
    # Split into chunks to avoid memory issues
    n = max(math.ceil(min(xs / maxs, ys / maxs)), 4)
    
    mmd = 0.
    for i in range(n):
        # Get chunk of x and y
        cur_x = x[round(i * xs / n): round((i + 1) * xs / n)]
        cur_y = y[round(i * ys / n): round((i + 1) * ys / n)]
        # Accumulate MMD
        mmd += _squared_mmd(cur_x, cur_y)
    
    return (mmd / n).item()

# Understanding `_calc_kid` — Chunked KID Calculation

## The Code

```python
#|export
def _calc_kid(x, y, maxs=50):
    xs, ys = x.shape[0], y.shape[0]
    n = max(math.ceil(min(xs / maxs, ys / maxs)), 4)
    
    mmd = 0.
    for i in range(n):
        cur_x = x[round(i * xs / n): round((i + 1) * xs / n)]
        cur_y = y[round(i * ys / n): round((i + 1) * ys / n)]
        mmd += _squared_mmd(cur_x, cur_y)
    
    return (mmd / n).item()
```

---

## Why Chunk at All?

The `_squared_mmd` function computes **kernel matrices** — pairwise comparisons between all samples. For `m` samples, this creates an `[m, m]` matrix:

| Samples (m) | Kernel Matrix Size | Memory (float32) |
|---|---|---|
| 50 | 50 × 50 = 2,500 | ~10 KB |
| 1,000 | 1,000 × 1,000 = 1,000,000 | ~4 MB |
| 10,000 | 10,000 × 10,000 = 100,000,000 | ~400 MB |

With 10,000 real image features, computing `k(x, x)` would require a 10,000 × 10,000 matrix — **400 MB of GPU memory** just for one kernel matrix (and we need three: `kxx`, `kyy`, `kxy`). Chunking avoids this by processing small subsets at a time.

---

## Line-by-Line Breakdown

### Step 1: Get Sample Counts

```python
xs, ys = x.shape[0], y.shape[0]
# xs = number of real image features     (e.g., 10000)
# ys = number of generated image features (e.g., 256)
```

### Step 2: Decide Number of Chunks

```python
n = max(math.ceil(min(xs / maxs, ys / maxs)), 4)
```

This is three operations nested together. Let's unpack with `xs=10000`, `ys=256`, `maxs=50`:

```
Step 2a:  xs / maxs = 10000 / 50 = 200
          ys / maxs = 256 / 50   = 5.12

Step 2b:  min(200, 5.12) = 5.12
          (Use the SMALLER ratio — no point making more chunks
           than the smaller set can support)

Step 2c:  math.ceil(5.12) = 6
          (Round up to ensure chunks don't exceed maxs)

Step 2d:  max(6, 4) = 6
          (Ensure at least 4 chunks for statistical stability)

Result:   n = 6 chunks
```

The logic ensures:
- **Each chunk** has at most ~`maxs` samples from the smaller set
- We always have **at least 4 chunks** (to get a reasonable average)

### Step 3: Loop Over Chunks

```python
mmd = 0.
for i in range(n):
    cur_x = x[round(i * xs / n): round((i + 1) * xs / n)]
    cur_y = y[round(i * ys / n): round((i + 1) * ys / n)]
    mmd += _squared_mmd(cur_x, cur_y)
```

#### The Slicing Formula

```python
x[round(i * xs / n) : round((i + 1) * xs / n)]
```

This divides `x` into `n` roughly equal chunks. With `xs=10000` and `n=6`:

| Chunk `i` | Start: `round(i * 10000/6)` | End: `round((i+1) * 10000/6)` | Size |
|---|---|---|---|
| 0 | `round(0)` = 0 | `round(1667)` = 1667 | 1667 |
| 1 | `round(1667)` = 1667 | `round(3333)` = 3333 | 1666 |
| 2 | `round(3333)` = 3333 | `round(5000)` = 5000 | 1667 |
| 3 | `round(5000)` = 5000 | `round(6667)` = 6667 | 1667 |
| 4 | `round(6667)` = 6667 | `round(8333)` = 8333 | 1666 |
| 5 | `round(8333)` = 8333 | `round(10000)` = 10000 | 1667 |

And `y` (256 samples) split into 6 chunks:

| Chunk `i` | Start | End | Size |
|---|---|---|---|
| 0 | 0 | 43 | 43 |
| 1 | 43 | 85 | 42 |
| 2 | 85 | 128 | 43 |
| 3 | 128 | 171 | 43 |
| 4 | 171 | 213 | 42 |
| 5 | 213 | 256 | 43 |

Each chunk of `y` has ~43 samples (under the `maxs=50` limit), keeping kernel matrices small.

#### What Happens Each Iteration

```
Chunk 0:  _squared_mmd(x[0:1667],    y[0:43])    → mmd₀
Chunk 1:  _squared_mmd(x[1667:3333],  y[43:85])   → mmd₁
Chunk 2:  _squared_mmd(x[3333:5000],  y[85:128])  → mmd₂
Chunk 3:  _squared_mmd(x[5000:6667],  y[128:171]) → mmd₃
Chunk 4:  _squared_mmd(x[6667:8333],  y[171:213]) → mmd₄
Chunk 5:  _squared_mmd(x[8333:10000], y[213:256]) → mmd₅

mmd = mmd₀ + mmd₁ + mmd₂ + mmd₃ + mmd₄ + mmd₅
```

### Step 4: Average and Return

```python
return (mmd / n).item()
```

Divide by `n` to get the **average MMD across chunks**, then `.item()` converts the single-element tensor to a plain Python float.

---

## Visual Summary

```
┌─────────────────────────────────────────────────────────────────┐
│                    _calc_kid FLOW                                │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  x (real features):      [████████████████████████] (10000)     │
│  y (generated features): [████] (256)                           │
│                                                                 │
│  Split into n=6 chunks:                                         │
│                                                                 │
│  x: [███|███|███|███|███|███]                                   │
│  y: [█|█|█|█|█|█]                                               │
│      ↓  ↓  ↓  ↓  ↓  ↓                                          │
│     mmd mmd mmd mmd mmd mmd                                    │
│      ₀   ₁   ₂   ₃   ₄   ₅                                    │
│      │   │   │   │   │   │                                      │
│      └───┴───┴───┼───┴───┘                                      │
│                  ↓                                               │
│           sum / 6 = KID                                         │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## Why `round()` Instead of Integer Division?

```python
# Using round():
x[round(i * xs / n) : round((i + 1) * xs / n)]

# Alternative with integer division:
# chunk_size = xs // n
# x[i * chunk_size : (i + 1) * chunk_size]
```

The `round()` approach distributes **remainders more evenly** across chunks. With integer division, the last chunk would absorb all the leftover samples and be disproportionately large. With `round()`, chunk sizes differ by at most 1 sample.

---

## Key Design Decisions

| Decision | Reason |
|---|---|
| `maxs=50` default | Keeps kernel matrices at most 50×50 — small enough for any GPU |
| `min(xs/maxs, ys/maxs)` | Chunk count driven by whichever set would exceed `maxs` first |
| `max(..., 4)` | At least 4 chunks ensures the average is statistically meaningful |
| Average (not sum) | Makes KID comparable regardless of how many chunks were used |
| `.item()` | Returns a Python float instead of a 0-dim tensor |

In [ ]:
# Calculate KID
_calc_kid(feats, feats2)

0.056

### 🎮 Interactive: the KID kernel arena — score every pair, then average

KID never computes μ or Σ. Instead it compares **all pairs of raw points** through a polynomial kernel k(a,b) = (a·b/d + 1)³, building three similarity tables: real·real, gen·gen, and real·gen. Then one subtraction — *within-cloud similarity minus cross-cloud similarity* — gives MMD², the KID score. This arena computes all of it **exactly, live**, on two clouds you can drag around.

**What to try:**
- Play stage ② and watch the three kernel matrices fill **cell by cell**, each cell's darkness = that pair's similarity. **Hover any cell** to see its full arithmetic with real numbers.
- Stage ③ is the famous unbiasedness trick: the diagonals of k(x,x′) and k(y,y′) get struck out (a point always matches itself — pure flattery).
- **Drag the teal cloud onto the orange one** — the three bars cancel and KID collapses to ≈ 0 (sometimes slightly *below* 0, which FID can never do).
- Stage ⑤ shows why the notebook's `_calc_kid` chops data into chunks of `maxs=50`: full 10000×10000 tables would eat all your memory.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/kid_kernel_arena.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/kid_kernel_arena.html", height="820px")

KID does not have nasty bias problem as FID does. But KID has very high variance. So when we call it multiple times with just like samples with different random seeds, we get different values. So, Jeremy has not found it useful at all.

**So, we din't have a good unbiased metric to compare the generated images with the real images.**

**What does the result mean?**

A KID of ~0.056 indicates some difference between distributions.

**KID vs FID:**
| | FID | KID |
|---|---|---|
| Assumption | Gaussian features | None |
| Bias | Biased for small samples | Unbiased |
| Computation | Needs matrix square root | Kernel computations |
| Interpretation | Absolute value | Relative comparison |

### 🎮 Interactive: the fairness test — catch FID lying

"FID is biased, KID is unbiased but noisy" — these are *statistical* claims, so let's run the actual experiment. Rig the game: draw **both** samples from the **same** distribution, so the true answer is exactly **0**. Repeat 40 times at each sample size n. Every dot is one full experiment (with FID and unbiased KID computed exactly).

**What to try:**
- Play stages ②–④: all 40 FID dots land **above** the truth line (bias — the notebook's real-vs-real FID of 6.61 is this effect in the wild), while KID dots scatter **around** it, some negative (like the notebook's −0.026).
- Slide **samples n** from 20 to 1000 and watch FID's bias shrink — but never vanish.
- Stage ⑤ plots the bias-vs-n curve directly; **click any dot on it** to switch the experiment to that n. The moral: *never compare FIDs measured at different sample counts.*

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/bias_variance_arena.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/bias_variance_arena.html", height="800px")

---

## Section 5: The ImageEval Class

Let's wrap everything into a convenient class for easy use.

In [ ]:
#|export
class ImageEval:
    """
    Convenient class for evaluating generated images with FID and KID.
    
    Usage:
    1. Create with a feature extractor model and real image DataLoaders
    2. Call .fid(generated_images) or .kid(generated_images) to evaluate
    
    Parameters:
    -----------
    model : nn.Module
        Feature extractor (e.g., classifier with head removed)
        It is a pre-trained model that is used to extract features from images.
    dls : DataLoaders
        DataLoaders with real images
    cbs : list
        Callbacks (e.g., DeviceCB)
    """
    def __init__(self, model, dls, cbs=None):
        # Create learner for feature extraction
        self.learn = TrainLearner(
            model, dls,
            loss_func=fc.noop,  # No loss needed
            cbs=cbs,
            opt_func=None       # No optimizer needed
        )
        
        # Extract features from all real images
        self.feats = self.learn.capture_preds()[0].float().cpu().squeeze()
        
        # Pre-compute statistics for real images
        self.stats = _calc_stats(self.feats)

    def get_feats(self, samp):
        """
        Extract features from generated samples.
        
        Parameters:
        -----------
        samp : torch.Tensor
            Generated images
            
        Returns:
        --------
        torch.Tensor : Features for generated images
        """
        # Create a temporary DataLoader with generated samples
        self.learn.dls = DataLoaders([], [(samp, tensor([0]))])
        return self.learn.capture_preds()[0].float().cpu().squeeze()

    def fid(self, samp):
        """
        Calculate FID for generated samples.
        
        Parameters:
        -----------
        samp : torch.Tensor
            Generated images
            
        Returns:
        --------
        float : FID score (lower is better)
        """
        gen_stats = _calc_stats(self.get_feats(samp)) # returns mean and covariance
        return _calc_fid(*self.stats, *gen_stats)
        # Here, self.stats is a tuple of (mean, covariance) for real images
        # and gen_stats is a tuple of (mean, covariance) for generated images
        # _calc_fid takes these four values and calculates the FID score
    
    def kid(self, samp):
        """
        Calculate KID for generated samples.
        
        Parameters:
        -----------
        samp : torch.Tensor
            Generated images
            
        Returns:
        --------
        float : KID score (lower is better)
        """
        return _calc_kid(self.feats, self.get_feats(samp))

**What does the code above do?**

Creates a convenient evaluation class:

1. **`__init__`**: 
   - Sets up the feature extractor
   - Pre-computes features and statistics for real images

2. **`get_feats`**: Extracts features from generated images

3. **`fid`**: Computes FID between generated and real images

4. **`kid`**: Computes KID between generated and real images

**Usage:**
```python
ie = ImageEval(feature_model, real_data_loaders, cbs=[DeviceCB()])
fid_score = ie.fid(generated_images)
kid_score = ie.kid(generated_images)
```

# Understanding `fc.noop` and `capture_preds()` in the ImageEval Class

## The Code in Question

```python
class ImageEval:
    def __init__(self, model, dls, cbs=None):
        # Create learner for feature extraction
        self.learn = TrainLearner(
            model, dls,
            loss_func=fc.noop,  # No loss needed
            cbs=cbs,
            opt_func=None       # No optimizer needed
        )
        
        # Extract features from all real images
        self.feats = self.learn.capture_preds()[0].float().cpu().squeeze()
        
        # Pre-compute statistics for real images
        self.stats = _calc_stats(self.feats)
```

---

# Part 1: Understanding `fc.noop`

## The Import

From the notebook:

```python
import fastcore.all as fc
```

So `fc` is the **fastcore** library (a utility library created by fast.ai).

## The Definition

`fc.noop` is a **"no operation" function** from fastcore - a function that accepts any arguments and does nothing:

```python
# fc.noop is literally a function that does nothing:
def noop(*args, **kwargs):
    pass
```

- `*args` - Accepts any positional arguments
- `**kwargs` - Accepts any keyword arguments
- `pass` - Does nothing and returns `None`

## Why Use `fc.noop` in ImageEval?

### The Context

The `ImageEval` class is designed for **evaluating generated images** using FID and KID metrics. It needs to:

1. Extract features from real images (once, during initialization)
2. Extract features from generated images (each time `.fid()` or `.kid()` is called)
3. Compare the feature distributions

**It does NOT need to:**
- Compute any loss
- Do backpropagation
- Update any weights

### The Problem

`TrainLearner` is designed for training neural networks, so it **requires** a `loss_func` parameter:

```python
TrainLearner(model, dls, loss_func=???, ...)
```

But in `ImageEval`, we only use `TrainLearner` to call `capture_preds()` for feature extraction - we never actually train anything!

### The Solution

Pass `fc.noop` as a placeholder that satisfies the requirement but does nothing:

```python
loss_func=fc.noop  # "I need to give you something, but I won't use it"
```

## Why Not Use `None`?

```python
# This might fail:
TrainLearner(model, dls, loss_func=None, ...)

# Because if TrainLearner ever tries to call it:
loss = self.loss_func(predictions, targets)  
# TypeError: 'NoneType' is not callable
```

`fc.noop` is a **real callable function**, so even if it gets called accidentally, it won't crash - it just does nothing and returns `None`.

```python
# fc.noop is safe to call with any arguments:
fc.noop()                           # OK - returns None
fc.noop(pred, target)               # OK - returns None  
fc.noop(a, b, c, reduction='mean')  # OK - returns None
```

---

# Part 2: Understanding `capture_preds()`

## The Line in Question

```python
self.feats = self.learn.capture_preds()[0].float().cpu().squeeze()
```

Let's break this down piece by piece.

---

## What Is `capture_preds()`?

`capture_preds()` is a **fastai/miniai method** defined on the `Learner` class (or `TrainLearner` in miniai). It's a convenience method that handles all the boilerplate code for running inference.

## What Does `capture_preds()` Do?

It runs the model on the **validation data** and captures:
1. **Model outputs** (predictions/features)
2. **True labels** (ground truth)

## Simplified Version of What It Looks Like Internally

```python
class Learner:
    def capture_preds(self):
        self.model.eval()                      # Set to evaluation mode
        preds, targets = [], []
        
        with torch.no_grad():                  # Disable gradient computation
            for xb, yb in self.dls.valid:      # Loop through validation data
                xb = xb.to(device)             # Move to GPU
                pred = self.model(xb)          # Forward pass
                preds.append(pred.cpu())       # Store prediction
                targets.append(yb)             # Store label
        
        return torch.cat(preds), torch.cat(targets)  # Concatenate all batches
```

## Why Use `capture_preds()` Instead of Manual Code?

It saves you from writing boilerplate:

```python
# ❌ Without capture_preds (manual way - tedious!):
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for xb, yb in validation_loader:
        xb = xb.cuda()
        pred = model(xb)
        all_preds.append(pred.cpu())
        all_labels.append(yb)
feats = torch.cat(all_preds)
labels = torch.cat(all_labels)

# ✅ With capture_preds (clean way - one line!):
feats, labels = learn.capture_preds()
```

**`capture_preds()` handles all the details for you:**
- Sets model to eval mode (`model.eval()`)
- Disables gradients (`torch.no_grad()`)
- Moves data to correct device (GPU/CPU)
- Concatenates results from all batches
- Returns clean tensors ready to use

## Step-by-Step Execution

```
┌─────────────────────────────────────────────────────────────────┐
│ capture_preds() execution:                                      │
│                                                                 │
│ 1. Switch model to eval mode (model.eval())                     │
│    • Disables dropout                                           │
│    • Uses running stats for BatchNorm                           │
│                                                                 │
│ 2. Disable gradient computation (torch.no_grad())               │
│    • Saves memory                                               │
│    • Speeds up computation                                      │
│                                                                 │
│ 3. Iterate over validation DataLoader:                          │
│    └── For each batch (xb, yb):                                │
│        ├── xb → moved to GPU                                   │
│        ├── pred = model(xb) → forward pass                     │
│        ├── preds.append(pred.cpu())                            │
│        └── targets.append(yb)                                  │
│                                                                 │
│ 4. Concatenate and return:                                      │
│    └── return (torch.cat(preds), torch.cat(targets))           │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## Breaking Down: `self.learn.capture_preds()[0].float().cpu().squeeze()`

### Step 1: `self.learn.capture_preds()`

Returns a tuple of two tensors:

```python
(predictions, labels) = self.learn.capture_preds()
#     │           │
#     │           └── Index [1]: Labels (ground truth)
#     └────────────── Index [0]: Model outputs (features)
```

### Step 2: `[0]`

Selects only the **predictions/features** (we don't need the labels for FID/KID):

```python
self.learn.capture_preds()[0]  # Just the features, ignore labels
```

### Step 3: `.float()`

Converts to **float32** precision:

```python
# The model might output in mixed precision (float16/bfloat16) for speed
# .float() converts to float32 for accurate FID/KID calculations

features.dtype  # Maybe torch.float16 or torch.bfloat16
features.float().dtype  # torch.float32
```

**Why?** FID/KID calculations involve computing means, covariances, and matrix square roots. These operations need float32 precision for numerical stability.

### Step 4: `.cpu()`

Moves the tensor from GPU to CPU:

```python
features.device  # cuda:0
features.cpu().device  # cpu
```

**Why?** 
- Frees GPU memory
- Some operations (like scipy's `sqrtm`) require CPU tensors
- Allows storing features without keeping GPU memory occupied

### Step 5: `.squeeze()`

Removes dimensions of size 1:

```python
# Possible shapes before squeeze:
features.shape = torch.Size([256, 512, 1, 1])  # Extra spatial dims from pooling
# or
features.shape = torch.Size([1, 256, 512])    # Extra batch dim

# After squeeze:
features.shape = torch.Size([256, 512])       # Clean 2D tensor!
```

**Visual Example:**

```
Before squeeze():
┌─────────────────────────────────────┐
│ Shape: [256, 512, 1, 1]             │
│                    ↑  ↑             │
│                    │  └── Size 1!   │
│                    └───── Size 1!   │
└─────────────────────────────────────┘
                    │
                    ▼ squeeze()
┌─────────────────────────────────────┐
│ Shape: [256, 512]                   │
│                                     │
│ Clean 2D matrix:                    │
│   256 images × 512 features         │
└─────────────────────────────────────┘
```

---

## Complete Flow Diagram

```
┌─────────────────────────────────────────────────────────────────┐
│          self.learn.capture_preds()[0].float().cpu().squeeze()  │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Step 1: capture_preds()                                        │
│  ────────────────────────                                       │
│  • Runs model on validation data                                │
│  • Returns (predictions, labels)                                │
│  • predictions shape: [N, 512, 1, 1] (maybe)                   │
│                    │                                            │
│                    ▼                                            │
│  Step 2: [0]                                                    │
│  ───────────                                                    │
│  • Select predictions only (index 0)                            │
│  • Ignore labels (index 1)                                      │
│  • Shape: [N, 512, 1, 1]                                       │
│                    │                                            │
│                    ▼                                            │
│  Step 3: .float()                                               │
│  ────────────────                                               │
│  • Convert to float32 precision                                 │
│  • For accurate FID/KID calculations                            │
│  • Shape: [N, 512, 1, 1] (dtype=float32)                       │
│                    │                                            │
│                    ▼                                            │
│  Step 4: .cpu()                                                 │
│  ──────────────                                                 │
│  • Move from GPU to CPU                                         │
│  • Free GPU memory                                              │
│  • Shape: [N, 512, 1, 1] (device=cpu)                          │
│                    │                                            │
│                    ▼                                            │
│  Step 5: .squeeze()                                             │
│  ──────────────────                                             │
│  • Remove dimensions of size 1                                  │
│  • [N, 512, 1, 1] → [N, 512]                                   │
│  • Clean 2D matrix ready for statistics!                        │
│                    │                                            │
│                    ▼                                            │
│  RESULT: self.feats                                             │
│  • Shape: [N, 512]                                              │
│  • N images × 512 features each                                 │
│  • Ready for FID/KID calculation!                               │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

# Part 3: Visual Explanation of ImageEval Flow

```
┌─────────────────────────────────────────────────────────────────┐
│                    NORMAL TRAINING FLOW                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   Images ──► Model ──► Predictions ──► loss_func() ──► Loss     │
│                                              │                  │
│                                              ▼                  │
│                                        Backpropagation          │
│                                              │                  │
│                                              ▼                  │
│                                        Update Weights           │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                    ImageEval FEATURE EXTRACTION                  │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   Images ──► Model ──► Features ──► capture_preds() returns     │
│                           │                                     │
│                           ▼                                     │
│                   self.feats = features                         │
│                   (stored for FID/KID calculation)              │
│                                                                 │
│   loss_func=fc.noop  → Never actually called!                   │
│   opt_func=None      → No optimizer needed                      │
│                                                                 │
│   ✗ No loss calculation                                         │
│   ✗ No backpropagation                                          │
│   ✗ No weight updates                                           │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

# Part 4: The Full ImageEval Class with Detailed Comments

```python
class ImageEval:
    """
    Convenient class for evaluating generated images with FID and KID.
    
    Usage:
    1. Create with a feature extractor model and real image DataLoaders
    2. Call .fid(generated_images) or .kid(generated_images) to evaluate
    """
    def __init__(self, model, dls, cbs=None):
        # Create learner for feature extraction
        # We use TrainLearner but we're NOT training - just extracting features
        self.learn = TrainLearner(
            model, dls,
            loss_func=fc.noop,  # Placeholder function that does nothing
                                # TrainLearner requires loss_func, but we don't need it
            cbs=cbs,            # Callbacks like DeviceCB() for GPU transfer
            opt_func=None       # No optimizer - we're not updating weights
        )
        
        # Extract features from all real images
        # capture_preds() returns (predictions, labels) tuple
        # [0] selects predictions only
        # .float() converts to float32 for numerical precision
        # .cpu() moves to CPU to free GPU memory
        # .squeeze() removes any size-1 dimensions like [N,512,1,1] → [N,512]
        self.feats = self.learn.capture_preds()[0].float().cpu().squeeze()
        
        # Pre-compute statistics (mean and covariance) for real images
        # This only needs to be done once, so we do it in __init__
        self.stats = _calc_stats(self.feats)

    def get_feats(self, samp):
        """
        Extract features from generated samples.
        
        Parameters:
        -----------
        samp : torch.Tensor
            Generated images, shape [N, C, H, W]
            
        Returns:
        --------
        torch.Tensor : Features for generated images, shape [N, feature_dim]
        """
        # Create a temporary DataLoader with generated samples
        # Empty [] for training (we don't train)
        # [(samp, tensor([0]))] for validation - one batch containing our samples
        # tensor([0]) is a dummy label (required by DataLoader but not used)
        self.learn.dls = DataLoaders([], [(samp, tensor([0]))])
        
        # Same processing as in __init__:
        # capture_preds()[0] = get predictions only
        # .float() = convert to float32
        # .cpu() = move to CPU
        # .squeeze() = remove size-1 dimensions
        return self.learn.capture_preds()[0].float().cpu().squeeze()

    def fid(self, samp):
        """
        Calculate FID (Fréchet Inception Distance) for generated samples.
        
        Parameters:
        -----------
        samp : torch.Tensor
            Generated images
            
        Returns:
        --------
        float : FID score (lower is better, 0 = identical distributions)
        """
        # Get features and statistics for generated images
        gen_stats = _calc_stats(self.get_feats(samp))
        
        # Compare with pre-computed real image statistics
        # self.stats = (mean_real, cov_real)
        # gen_stats = (mean_gen, cov_gen)
        # *self.stats unpacks to mean_real, cov_real
        # *gen_stats unpacks to mean_gen, cov_gen
        return _calc_fid(*self.stats, *gen_stats)
    
    def kid(self, samp):
        """
        Calculate KID (Kernel Inception Distance) for generated samples.
        
        Parameters:
        -----------
        samp : torch.Tensor
            Generated images
            
        Returns:
        --------
        float : KID score (lower is better, 0 = identical distributions)
        """
        # KID compares features directly (not statistics)
        # self.feats = real image features
        # self.get_feats(samp) = generated image features
        return _calc_kid(self.feats, self.get_feats(samp))
```

---

# Summary Tables

## `fc.noop` Summary

| Question | Answer |
|----------|--------|
| What is `fc`? | `fastcore` library (`import fastcore.all as fc`) |
| What is `noop`? | A function that does nothing: `def noop(*args, **kwargs): pass` |
| Why use it here? | `TrainLearner` requires `loss_func`, but `ImageEval` doesn't need loss |
| Is `fc.noop` ever called? | Likely not in this usage - it's just a placeholder |
| Why not `None`? | `None` isn't callable; `fc.noop` is safe if accidentally called |

## `capture_preds()` Summary

| Question | Answer |
|----------|--------|
| What is `capture_preds()`? | A fastai/miniai method that runs inference on validation data |
| What does it return? | Tuple of `(predictions, labels)` |
| Why `[0]}? | Select only predictions, ignore labels |
| Why `.float()`? | Convert to float32 for accurate FID/KID calculations |
| Why `.cpu()`? | Move to CPU, free GPU memory |
| Why `.squeeze()`? | Remove size-1 dimensions for clean `[N, 512]` shape |

## Method Chain Summary

| Step | Code | What It Does |
|------|------|--------------|
| 1 | `capture_preds()` | Run model on validation data, return (preds, labels) |
| 2 | `[0]` | Select predictions only |
| 3 | `.float()` | Convert to float32 precision |
| 4 | `.cpu()` | Move from GPU to CPU |
| 5 | `.squeeze()` | Remove dimensions of size 1 |

---

# Key Takeaways

1. **`fc.noop`** is a placeholder function that lets us use `TrainLearner` for inference without needing a real loss function.

2. **`capture_preds()`** handles all the boilerplate for inference: eval mode, no_grad, device transfer, and batch concatenation.

3. **The method chain** `.float().cpu().squeeze()` ensures we get clean, properly-formatted features ready for FID/KID calculation.

4. **`ImageEval`** pre-computes real image features once in `__init__`, then efficiently compares against generated images in `.fid()` and `.kid()`.

In [ ]:
# Create the ImageEval evaluator
ie = ImageEval(model, learn.dls, cbs=[DeviceCB()])

In [ ]:
%%time
# Calculate FID for our generated samples
ie.fid(s)

33.90

In [ ]:
%%time
# Calculate KID for our generated samples
ie.kid(s)

0.056

KID is normally smaller than FID.

### FID and KID Over the Denoising Process

Let's see how quality improves during the DDPM sampling process.

In [ ]:
# Calculate FID at different timesteps
# xs contains timestep indices to evaluate
xs = L.range(0, 1000, 50) + [975, 990, 999]

# samples[i] is the image at timestep n_steps-1-i
# We clamp and scale to match expected range
plt.plot(xs, [ie.fid(samples[i].clamp(-0.5, 0.5) * 2) for i in xs])
plt.xlabel('Denoising step')
plt.ylabel('FID')
plt.title('FID vs Denoising Progress');

# Understanding `xs` in the FID vs Denoising Progress Plot

## The Code

```python
# Calculate FID at different timesteps
# xs contains timestep indices to evaluate
xs = L.range(0, 1000, 50) + [975, 990, 999]

# samples[i] is the image at timestep n_steps-1-i
# We clamp and scale to match expected range
plt.plot(xs, [ie.fid(samples[i].clamp(-0.5, 0.5) * 2) for i in xs])
plt.xlabel('Denoising step')
plt.ylabel('FID')
plt.title('FID vs Denoising Progress');
```

---

## What is `L`?

`L` is from **fastcore** (imported at the top of the notebook):

```python
from fastcore.foundation import L
```

`L` is an enhanced list class from fastcore that provides additional functionality like `.range()`.

---

## Breaking Down `L.range(0, 1000, 50)`

This creates a list of numbers from 0 to 1000, stepping by 50:

```python
L.range(0, 1000, 50)
# Result: [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 
#          500, 550, 600, 650, 700, 750, 800, 850, 900, 950]
```

**Note:** Just like Python's `range()`, it stops **before** 1000, so 1000 is not included.

---

## Breaking Down `+ [975, 990, 999]`

This adds three more values at the end to capture the **final denoising steps** with finer granularity:

```python
L.range(0, 1000, 50) + [975, 990, 999]
# Result: [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 
#          500, 550, 600, 650, 700, 750, 800, 850, 900, 950,
#          975, 990, 999]
```

---

## Why These Specific Values?

```
┌─────────────────────────────────────────────────────────────────┐
│                    DENOISING PROCESS (1000 steps)                │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Step 0        Step 500       Step 950    Step 999              │
│    │              │              │           │                  │
│    ▼              ▼              ▼           ▼                  │
│  ┌────┐        ┌────┐        ┌────┐      ┌────┐                │
│  │░░░░│   →    │▒▒▒▒│   →    │▓▓▓▓│  →   │████│                │
│  │░░░░│        │▒▒▒▒│        │▓▓▓▓│      │████│                │
│  └────┘        └────┘        └────┘      └────┘                │
│   Pure          Partial       Almost      Final                 │
│   Noise         Denoised      Clean       Image                 │
│                                                                 │
│  xs samples at:                                                 │
│  [0, 50, 100, 150, ... 900, 950] ← Every 50 steps (coarse)     │
│  [975, 990, 999]                 ← Final steps (fine detail)   │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Why Add [975, 990, 999]?

The **final denoising steps** are where the most important details emerge. By adding these values, we can see:

- How FID improves in the **critical final phase**
- Whether the model converges smoothly or has sudden changes
- The exact quality at the **very last step** (999)

---

## Complete `xs` Values

```python
xs = [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 
      500, 550, 600, 650, 700, 750, 800, 850, 900, 950,
      975, 990, 999]

len(xs)  # 23 values total
```

| Range | Values | Purpose |
|-------|--------|---------|
| 0-950 | Every 50 steps | Coarse sampling of denoising progress |
| 975, 990, 999 | Final 3 points | Fine detail in critical final phase |

---

## How `xs` is Used in the Plot

```python
plt.plot(xs, [ie.fid(samples[i].clamp(-0.5, 0.5) * 2) for i in xs])
```

For each value `i` in `xs`:

| Step | Code | What It Does |
|------|------|--------------|
| 1 | `samples[i]` | Get the image at denoising step `i` |
| 2 | `.clamp(-0.5, 0.5)` | Clamp pixel values to range [-0.5, 0.5] |
| 3 | `* 2` | Scale to range [-1, 1] to match training data |
| 4 | `ie.fid(...)` | Calculate FID score for this image |
| 5 | `plt.plot(xs, [...])` | Plot the point (step, FID) |

---

## What Does `samples[i]` Contain?

During the DDPM sampling process, `samples` is a list that stores the image at each denoising step:

```python
samples = sample(smodel, (256, 1, 32, 32), alpha, alphabar, sigma, n_steps)
# samples[0]   = image after step 0   (still very noisy)
# samples[500] = image after step 500 (partially denoised)
# samples[999] = image after step 999 (final clean image)
```

---

## Why `.clamp(-0.5, 0.5) * 2`?

This ensures the generated images are in the correct range for the feature extractor:

```
┌─────────────────────────────────────────────────────────────────┐
│                    VALUE RANGE ADJUSTMENT                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Raw samples output:     might be outside [-0.5, 0.5]           │
│                               │                                 │
│                               ▼                                 │
│  .clamp(-0.5, 0.5):      force into range [-0.5, 0.5]          │
│                               │                                 │
│                               ▼                                 │
│  * 2:                    scale to range [-1, 1]                 │
│                               │                                 │
│                               ▼                                 │
│  Final range:            [-1, 1] (matches training data)        │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

The feature extractor (classifier) was trained on images in the range [-1, 1], so we need to match that.

---

## Expected Plot Result

```
FID
 │
 │  ●                                    
 │    ●                                  
 │      ●                                
 │        ●  ●                           
 │             ●  ●                      
 │                  ●  ●  ●              
 │                          ●  ●  ●      
 │                                 ●  ●  ● ← Fine detail at end
 └────────────────────────────────────────── Denoising step
    0   100  200  300  400  500  600  700  800  900  999
```

**FID decreases** as denoising progresses because:
- **Early steps (0-200):** mostly noise → high FID (very different from real images)
- **Middle steps (300-700):** emerging structure → medium FID
- **Later steps (800-999):** cleaner images → lower FID (more similar to real images)

---

## Summary

| Question | Answer |
|----------|--------|
| What is `L`? | Enhanced list class from fastcore |
| What does `L.range(0, 1000, 50)` produce? | [0, 50, 100, ..., 900, 950] |
| Why add [975, 990, 999]? | Finer sampling of critical final denoising steps |
| Total number of points? | 23 |
| What is `samples[i]`? | The image at denoising step `i` |
| Why `.clamp(-0.5, 0.5) * 2`? | Scale to [-1, 1] range to match training data |
| What does the plot show? | FID score at different stages of denoising |
| Expected trend? | FID decreases as denoising progresses (images get cleaner) |

**What does this plot show?**

FID decreases as denoising progresses:
- Early steps (high FID): Images are still noisy
- Later steps (low FID): Images look more like real data

This confirms that the denoising process is working!

In [ ]:
# Same for KID
xs = L.range(0, 1000, 50) + [975, 990, 999]
plt.plot(xs, [ie.kid(samples[i].clamp(-0.5, 0.5) * 2) for i in xs])
plt.xlabel('Denoising step')
plt.ylabel('KID')
plt.title('KID vs Denoising Progress');

### 🎮 Interactive: the denoising scoreboard — quality is a curve, not a number

The plots above checkpoint the sampler at steps `0, 50, …, 950, 975, 990, 999` and score each one. This scoreboard turns that into a journey you can scrub: a step slider, both metric curves (FID and KID told the same story — strong evidence the improvement is real), and a thumbnail showing what the sample looks like at that instant.

**What to try:**
- **Drag the step scrubber** or simply **click/drag anywhere on the curves** — the cursor, thumbnail, and readout follow, and the stage chips + code panel sync to whichever phase of the journey you're in.
- Ride the stages: ① pure noise scores ~330 → ② the productive cliff → ③ diminishing returns → ④ the last-25-step polish (why the notebook adds extra checkpoints at 975/990/999) → ⑤ how to read the final 33.8 honestly.
- Toggle **log scale** to stretch out the flat tail and see that steps 950→999 still buy a little.
- Note the endgame lesson hiding in the flat tail: stopping at step 975 would cost almost nothing — useful when sampling time matters.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/denoise_scoreboard.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/denoise_scoreboard.html", height="820px")

### Sanity Check: FID/KID of Real Images

What if we evaluate real images? The scores should be very low!

In [ ]:
# FID of real images (should be low)
# xb is a batch of real images
ie.fid(xb)

6.61

In [ ]:
# KID of real images (should be near zero or slightly negative due to estimation variance)
ie.kid(xb)

-0.026

**What do these results show?**

- **FID of real images ≈ 6.6**: Lower than generated images (~34)
- **KID of real images ≈ -0.03**: Near zero (negative due to estimation variance)

This confirms our metrics are working correctly - real images score better than generated ones!

---

## Section 6: Using InceptionV3 (Standard FID)

The standard FID uses features from **InceptionV3** trained on ImageNet, not a custom classifier.

**Why InceptionV3?**
- Standardized: Everyone uses the same model, making results comparable
- General features: Trained on diverse ImageNet, captures general visual concepts
- Industry standard: Published FID scores typically use InceptionV3

In [ ]:
# Import InceptionV3 from pytorch_fid library
# Install with: pip install pytorch-fid
from pytorch_fid.inception import InceptionV3

In [ ]:
# Demo: tensor.repeat() duplicates a tensor
# We'll need this to convert grayscale to RGB
a = tensor([1, 2, 3])
a.repeat((3, 1))  # Repeat 3 times along first dimension

**Why do we need repeat?**

InceptionV3 expects RGB images (3 channels), but Fashion MNIST is grayscale (1 channel). We'll repeat the grayscale channel 3 times to create a "fake" RGB image.

In [ ]:
class IncepWrap(nn.Module):
    """
    Wrapper around InceptionV3 that:
    1. Converts grayscale to RGB by repeating channels
    2. Returns features from the standard FID layer
    """
    def __init__(self):
        super().__init__()
        # resize_input=True handles resizing to Inception's expected size
        self.m = InceptionV3(resize_input=True)
    
    def forward(self, x):
        # Convert grayscale to RGB: repeat channel 3 times
        # (B, 1, H, W) -> (B, 3, H, W)
        rgb = x.repeat(1, 3, 1, 1)
        # Get features from Inception (returns a list, we want first element)
        return self.m(rgb)[0]

**What does the code above do?**

Creates a wrapper that:
1. **Converts grayscale to RGB**: `x.repeat(1, 3, 1, 1)` copies the single channel 3 times
2. **Runs InceptionV3**: Gets features from the standard layer
3. **Returns features**: Shape will be (batch, 2048)

### 🎮 Interactive: swap the ruler — custom classifier vs InceptionV3

You're about to see the *same* generated images score **63.8** with InceptionV3 after scoring **33.8** with our custom classifier. Neither number is wrong: FID is only defined *relative to a feature extractor*, and this lab walks through everything that changes when you swap rulers — the channel repeat, the 299×299 resize, 512-D vs 2048-D features, and finally the two incomparable numbers side by side.

**What to try:**
- Toggle between **custom 512-D** and **InceptionV3 2048-D** and watch the pipeline itself change shape — the `x.repeat(1,3,1,1)` and resize stations only exist on the Inception path.
- Stage ① animates the grayscale sheet being copied into R = G = B; stage ② stretches 28px → 299px live.
- Stage ③ compares the feature spaces: Σ balloons from 512×512 to 2048×2048 — 16× more numbers to estimate, hence even harsher small-sample bias (real-vs-real jumps from 6.6 to 27.9!).
- Stage ⑤ is the one rule to leave with: FIDs are comparable **only** when extractor, preprocessing, and sample count all match.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/inception_switch_lab.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/inception_switch_lab.html", height="800px")

In [ ]:
# Reload data
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=fc.defaults.cpus)

In [ ]:
# Create ImageEval with InceptionV3
ie = ImageEval(IncepWrap(), dls, cbs=[DeviceCB()])

In [ ]:
%%time
# FID with InceptionV3
ie.fid(s)

63.81

In [ ]:
# FID of real images with InceptionV3
ie.fid(xb)

27.91

In [ ]:
%%time
# KID with InceptionV3
ie.kid(s)

0.05643

In [ ]:
# KID of real images with InceptionV3
ie.kid(xb)

-0.02641

# FID with Custom Classifier vs InceptionV3: What's the Difference?

## Overview

The notebook demonstrates **two different approaches** to calculating FID:

| Section | Feature Extractor | Description |
|---------|------------------|-------------|
| Sections 2-5 | **Custom Classifier** (trained on Fashion MNIST) | Uses a ResNet-style model trained specifically on the dataset |
| Section 6 | **InceptionV3** (trained on ImageNet) | The industry-standard feature extractor for FID |

**Both use DDPM-generated images** - the difference is only in HOW the features are extracted, not WHAT images are evaluated.

---

## CRITICAL CLARIFICATION: FID Requires Features from BOTH Real and Generated Images

FID compares **two distributions**:
1. Features from **real images** (Fashion MNIST validation set)
2. Features from **generated images** (DDPM samples)

**The SAME feature extractor must be used for BOTH** to make a fair comparison.

### Sections 2-5: Custom Classifier Extracts Features from BOTH

```python
# Feature extractor: Custom classifier trained on Fashion MNIST
model = torch.load('models/data_aug2.pkl')
del(learn.model[8])  # Remove classification head
del(learn.model[7])

# Create ImageEval
ie = ImageEval(model, learn.dls, cbs=[DeviceCB()])
#              ↑            ↑
#              │            └── Real images (Fashion MNIST validation set)
#              └─────────────── Feature extractor

# Inside ImageEval.__init__:
# self.feats = features from REAL images (extracted using custom classifier)

# When we call:
ie.fid(s)  # s = DDPM-generated images
# It extracts features from GENERATED images (using same custom classifier)
# Then compares the two distributions
```

### Section 6: InceptionV3 Extracts Features from BOTH

```python
# Feature extractor: InceptionV3 (pretrained on ImageNet)
ie = ImageEval(IncepWrap(), dls, cbs=[DeviceCB()])
#              ↑            ↑
#              │            └── Real images (Fashion MNIST validation set)
#              └─────────────── Feature extractor

# Inside ImageEval.__init__:
# self.feats = features from REAL images (extracted using InceptionV3)

# When we call:
ie.fid(s)  # s = DDPM-generated images
# It extracts features from GENERATED images (using same InceptionV3)
# Then compares the two distributions
```

---

## Visual Summary: Same Feature Extractor for Both Real and Generated

```
┌─────────────────────────────────────────────────────────────────┐
│                    SECTIONS 2-5: CUSTOM CLASSIFIER               │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   REAL IMAGES                      GENERATED IMAGES             │
│   (Fashion MNIST)                  (from DDPM)                  │
│        │                                │                       │
│        ▼                                ▼                       │
│   ┌──────────────────────────────────────────────┐             │
│   │         CUSTOM CLASSIFIER (512-dim)          │             │
│   │         (same model for both!)               │             │
│   └──────────────────────────────────────────────┘             │
│        │                                │                       │
│        ▼                                ▼                       │
│   Real Features                   Generated Features            │
│   [N, 512]                        [M, 512]                     │
│        │                                │                       │
│        └────────────┬───────────────────┘                       │
│                     ▼                                           │
│              ┌─────────────┐                                    │
│              │ COMPARE FID │                                    │
│              └─────────────┘                                    │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                    SECTION 6: InceptionV3                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   REAL IMAGES                      GENERATED IMAGES             │
│   (Fashion MNIST)                  (from DDPM)                  │
│        │                                │                       │
│        ▼                                ▼                       │
│   ┌──────────────────────────────────────────────┐             │
│   │           InceptionV3 (2048-dim)             │             │
│   │           (same model for both!)             │             │
│   └──────────────────────────────────────────────┘             │
│        │                                │                       │
│        ▼                                ▼                       │
│   Real Features                   Generated Features            │
│   [N, 2048]                       [M, 2048]                    │
│        │                                │                       │
│        └────────────┬───────────────────┘                       │
│                     ▼                                           │
│              ┌─────────────┐                                    │
│              │ COMPARE FID │                                    │
│              └─────────────┘                                    │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Key Insight

**The feature extractor must be IDENTICAL for both real and generated images** - that's the only way to fairly compare the distributions!

---

## Section 2-5: Custom Classifier Approach

### The Feature Extractor

```python
# Load a pre-trained Fashion MNIST classifier
model = torch.load('models/data_aug2.pkl')

# Remove the classification head to get features
del(learn.model[8])  # Remove final classification layer
del(learn.model[7])  # Remove pooling/activation layer

# Now model outputs 512-dim features instead of class probabilities
```

### What This Model Is

- A **ResNet-style CNN** trained to classify Fashion MNIST (10 classes)
- Trained specifically on **Fashion MNIST** data
- Outputs **512-dimensional features** after removing the classification head

### How It's Used

```python
# Create evaluator with custom classifier
ie = ImageEval(model, learn.dls, cbs=[DeviceCB()])

# Evaluate DDPM-generated images
ie.fid(s)  # s = generated samples from DDPM
ie.kid(s)
```

### Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│              CUSTOM CLASSIFIER (Fashion MNIST)                   │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Input: [B, 1, 32, 32]  (grayscale Fashion MNIST)              │
│         │                                                       │
│         ▼                                                       │
│  ┌─────────────────────┐                                        │
│  │ ResNet-style layers │                                        │
│  │ (trained on Fashion │                                        │
│  │  MNIST)             │                                        │
│  └─────────────────────┘                                        │
│         │                                                       │
│         ▼                                                       │
│  Output: [B, 512]  (512-dim features)                          │
│                                                                 │
│  Features capture: clothing-specific patterns                   │
│  (shirts, pants, shoes, etc.)                                   │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## Section 6: InceptionV3 Approach

### The Feature Extractor

```python
from pytorch_fid.inception import InceptionV3

class IncepWrap(nn.Module):
    def __init__(self):
        super().__init__()
        self.m = InceptionV3(resize_input=True)
    
    def forward(self, x):
        # Convert grayscale to RGB
        rgb = x.repeat(1, 3, 1, 1)
        return self.m(rgb)[0]
```

### What This Model Is

- **InceptionV3** pre-trained on **ImageNet** (1000 classes, millions of images)
- The **industry standard** for FID calculation
- Outputs **2048-dimensional features** from the pool3 layer

### How It's Used

```python
# Create evaluator with InceptionV3
ie = ImageEval(IncepWrap(), dls, cbs=[DeviceCB()])

# Evaluate the SAME DDPM-generated images
ie.fid(s)  # s = same generated samples from DDPM
ie.kid(s)
```

### Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                    InceptionV3 (ImageNet)                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Input: [B, 1, 32, 32]  (grayscale Fashion MNIST)              │
│         │                                                       │
│         ▼                                                       │
│  ┌─────────────────────┐                                        │
│  │ x.repeat(1,3,1,1)   │  Convert grayscale → RGB               │
│  └─────────────────────┘                                        │
│         │                                                       │
│         ▼                                                       │
│  [B, 3, 32, 32]  (fake RGB)                                    │
│         │                                                       │
│         ▼                                                       │
│  ┌─────────────────────┐                                        │
│  │ resize_input=True   │  Resize to 299×299 (Inception size)   │
│  └─────────────────────┘                                        │
│         │                                                       │
│         ▼                                                       │
│  ┌─────────────────────┐                                        │
│  │ InceptionV3 layers  │                                        │
│  │ (trained on ImageNet│                                        │
│  │  1000 classes)      │                                        │
│  └─────────────────────┘                                        │
│         │                                                       │
│         ▼                                                       │
│  Output: [B, 2048]  (2048-dim features)                        │
│                                                                 │
│  Features capture: general visual concepts                      │
│  (edges, textures, shapes, objects)                            │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## Key Differences

| Aspect | Custom Classifier | InceptionV3 |
|--------|------------------|-------------|
| **Training Data** | Fashion MNIST (60K images, 10 classes) | ImageNet (1.2M images, 1000 classes) |
| **Feature Dim** | 512 | 2048 |
| **Input Size** | 32×32 grayscale | 299×299 RGB |
| **Specificity** | Dataset-specific features | General visual features |
| **Comparability** | Only comparable within your experiments | Comparable with published results |
| **Preprocessing** | None needed | Grayscale→RGB conversion, resizing |

---

## What Images Are Being Evaluated?

**BOTH approaches evaluate the SAME DDPM-generated images!**

```python
# Generate images with DDPM (this happens ONCE, in Section 3)
samples = sample(smodel, (256, 1, 32, 32), alpha, alphabar, sigma, n_steps)
s = samples[-1] * 2 - 1  # Final denoised images

# Section 5: Evaluate with custom classifier
ie_custom = ImageEval(model, learn.dls, cbs=[DeviceCB()])
fid_custom = ie_custom.fid(s)  # Same 's'

# Section 6: Evaluate with InceptionV3
ie_inception = ImageEval(IncepWrap(), dls, cbs=[DeviceCB()])
fid_inception = ie_inception.fid(s)  # Same 's'
```

The only difference is **HOW features are extracted** from those images.

---

## Visual Comparison: Two Approaches to the Same Problem

```
┌─────────────────────────────────────────────────────────────────┐
│                    SAME GENERATED IMAGES                         │
│                                                                 │
│                    ┌─────────────┐                              │
│                    │ DDPM Model  │                              │
│                    │ generates   │                              │
│                    │ 256 images  │                              │
│                    └──────┬──────┘                              │
│                           │                                     │
│                           ▼                                     │
│                    ┌─────────────┐                              │
│                    │  Generated  │                              │
│                    │   Images    │                              │
│                    │ [256,1,32,32]│                             │
│                    └──────┬──────┘                              │
│                           │                                     │
│            ┌──────────────┴──────────────┐                      │
│            │                             │                      │
│            ▼                             ▼                      │
│   ┌─────────────────┐          ┌─────────────────┐             │
│   │ Custom Classifier│          │   InceptionV3   │             │
│   │ (512-dim feats) │          │ (2048-dim feats)│             │
│   └────────┬────────┘          └────────┬────────┘             │
│            │                             │                      │
│            ▼                             ▼                      │
│   ┌─────────────────┐          ┌─────────────────┐             │
│   │ FID ≈ 34        │          │ FID ≈ ???       │             │
│   │ (dataset-specific│          │ (comparable to  │             │
│   │  metric)        │          │  published FID) │             │
│   └─────────────────┘          └─────────────────┘             │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## Why Use Both?

### Custom Classifier (Sections 2-5)

**Pros:**
- Faster (smaller model, no resizing)
- Features specifically tuned for Fashion MNIST
- Good for learning/experimentation

**Cons:**
- Results not comparable to other papers
- Only works for this specific dataset

### InceptionV3 (Section 6)

**Pros:**
- **Industry standard** - results comparable to published papers
- Works for any image dataset
- Well-tested and validated

**Cons:**
- Slower (larger model, needs resizing)
- Requires grayscale→RGB conversion for non-RGB datasets
- Features may not be optimal for specific domains

---

## Did We Use DDPM for Both?

**YES!** Both sections evaluate images generated by the **same DDPM model**.

```python
# This DDPM model is used throughout the notebook
smodel = torch.load('models/fashion_ddpm_mp.pkl').cuda()

# Generate samples (done once)
samples = sample(smodel, (256, 1, 32, 32), alpha, alphabar, sigma, n_steps)
s = samples[-1] * 2 - 1

# 's' is evaluated by BOTH:
# - Custom classifier in Section 5
# - InceptionV3 in Section 6
```

---

## Complete Picture: What Happens in Each Section

### Sections 2-5 Flow

```
┌─────────────────────────────────────────────────────────────────┐
│                    SECTIONS 2-5 COMPLETE FLOW                    │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  STEP 1: Load Custom Classifier                                 │
│  ───────────────────────────────                                │
│  model = torch.load('models/data_aug2.pkl')                    │
│  del(learn.model[8])  # Remove classification head              │
│  del(learn.model[7])                                           │
│                                                                 │
│  STEP 2: Create ImageEval                                       │
│  ────────────────────────                                       │
│  ie = ImageEval(model, learn.dls, cbs=[DeviceCB()])            │
│       │                                                         │
│       └── This extracts features from REAL images              │
│           using the custom classifier and stores them           │
│                                                                 │
│  STEP 3: Generate Images with DDPM                              │
│  ─────────────────────────────────                              │
│  samples = sample(smodel, ...)                                 │
│  s = samples[-1] * 2 - 1  # Generated images                   │
│                                                                 │
│  STEP 4: Calculate FID                                          │
│  ─────────────────────                                          │
│  ie.fid(s)                                                     │
│       │                                                         │
│       └── This extracts features from GENERATED images         │
│           using the SAME custom classifier                      │
│           Then compares with pre-computed real features         │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Section 6 Flow

```
┌─────────────────────────────────────────────────────────────────┐
│                    SECTION 6 COMPLETE FLOW                       │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  STEP 1: Create InceptionV3 Wrapper                             │
│  ──────────────────────────────────                             │
│  class IncepWrap wraps InceptionV3 with:                       │
│    - Grayscale → RGB conversion                                │
│    - Automatic resizing to 299×299                             │
│                                                                 │
│  STEP 2: Create ImageEval                                       │
│  ────────────────────────                                       │
│  ie = ImageEval(IncepWrap(), dls, cbs=[DeviceCB()])            │
│       │                                                         │
│       └── This extracts features from REAL images              │
│           using InceptionV3 and stores them                     │
│                                                                 │
│  STEP 3: Use SAME Generated Images from DDPM                    │
│  ───────────────────────────────────────────                    │
│  s = samples[-1] * 2 - 1  # Same generated images as before    │
│                                                                 │
│  STEP 4: Calculate FID                                          │
│  ─────────────────────                                          │
│  ie.fid(s)                                                     │
│       │                                                         │
│       └── This extracts features from GENERATED images         │
│           using the SAME InceptionV3                            │
│           Then compares with pre-computed real features         │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## Summary Tables

### What Gets Compared

| | Real Images | Generated Images | Feature Extractor |
|---|-------------|------------------|-------------------|
| **Sections 2-5** | Fashion MNIST validation | DDPM samples | Custom Classifier (512-dim) |
| **Section 6** | Fashion MNIST validation | DDPM samples | InceptionV3 (2048-dim) |

### Final Summary

| Question | Answer |
|----------|--------|
| What generates the images? | DDPM (same model for both sections) |
| What's different? | The feature extractor used for FID calculation |
| Section 2-5 feature extractor | Custom ResNet trained on Fashion MNIST (512-dim) |
| Section 6 feature extractor | InceptionV3 trained on ImageNet (2048-dim) |
| Does each approach extract features from real images? | **YES** - during `ImageEval.__init__` |
| Does each approach extract features from generated images? | **YES** - when calling `ie.fid(s)` |
| Must the same extractor be used for both? | **YES** - fair comparison requires identical feature extraction |
| Which is "correct"? | InceptionV3 is the standard; custom is for learning |
| Why show both? | Educational: understand FID concept first, then use standard |
| Are results comparable? | Only InceptionV3 results can be compared to published FID scores |

**Comparing Results:**

| Metric | Custom Classifier | InceptionV3 |
|--------|------------------|-------------|
| FID (generated) | ~34 | ~64 |
| FID (real) | ~6.6 | ~28 |
| KID (generated) | ~0.056 | ~0.011 |
| KID (real) | ~-0.03 | ~0 |

**Why are InceptionV3 FID scores higher?**

- InceptionV3 was trained on natural images (ImageNet), not fashion items
- The features may be less relevant for Fashion MNIST
- However, using InceptionV3 makes scores comparable with other papers

# Understanding `ie.fid(xb)` - The "Sanity Check"

## The Confusion

When we see this code:

```python
# FID of real images (should be low)
# xb is a batch of real images
ie.fid(xb)
```

It seems like we're comparing real images against real images. If `ImageEval` stores features from real images, and `xb` is also real images, are we comparing the same images against themselves?

**No! They are DIFFERENT sets of real images!**

---

## Key Insight: `self.feats` vs `xb` are DIFFERENT Sets

### What `self.feats` Contains

When `ImageEval` is created:

```python
ie = ImageEval(model, learn.dls, cbs=[DeviceCB()])
```

Inside `__init__`:

```python
# capture_preds() runs on the VALIDATION set (dls.valid)
self.feats = self.learn.capture_preds()[0].float().cpu().squeeze()
```

`capture_preds()` extracts features from the **validation set** (`dls.valid`), which contains **10,000 images** (the full Fashion MNIST validation set).

### What `xb` Contains

Earlier in the notebook:

```python
# ONE BATCH from the TRAINING set!
b = xb, yb = next(iter(dls.train))
```

`xb` is just **one batch (512 images)** from the **training set**, NOT the validation set!

---

## What `ie.fid(xb)` Actually Compares

```
┌─────────────────────────────────────────────────────────────────┐
│                    ie.fid(xb) COMPARISON                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   self.feats (stored in ImageEval)     xb (passed to fid())    │
│   ─────────────────────────────────    ─────────────────────   │
│   • From VALIDATION set                • From TRAINING set      │
│   • 10,000 images                      • 512 images (1 batch)   │
│   • Extracted during __init__          • Extracted when called  │
│                                                                 │
│        │                                      │                 │
│        ▼                                      ▼                 │
│   ┌──────────────────────────────────────────────┐             │
│   │            Feature Extractor                 │             │
│   └──────────────────────────────────────────────┘             │
│        │                                      │                 │
│        ▼                                      ▼                 │
│   Real Features                         Real Features           │
│   (validation)                          (training batch)        │
│   [10000, 512]                          [512, 512]             │
│        │                                      │                 │
│        └──────────────┬───────────────────────┘                 │
│                       ▼                                         │
│                ┌─────────────┐                                  │
│                │ COMPARE FID │                                  │
│                └─────────────┘                                  │
│                       │                                         │
│                       ▼                                         │
│                  FID ≈ 6.6                                      │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### The Comparison

| | `self.feats` | `xb` |
|---|--------------|------|
| **Source** | Validation set | Training set |
| **Number of images** | 10,000 | 512 (one batch) |
| **When extracted** | During `ImageEval.__init__` | When `ie.fid(xb)` is called |
| **Type of images** | Real (Fashion MNIST) | Real (Fashion MNIST) |

**Both are REAL images, just from DIFFERENT splits of the dataset!**

---

## Why FID ≈ 6.6 (Not Zero)?

Even though **both are real images**, FID is not exactly zero because:

1. **Different splits**: Validation vs Training sets have slightly different distributions
2. **Sample size mismatch**: Only 512 training images vs 10,000 validation images
3. **Estimation variance**: Statistical estimation of mean and covariance has some noise
4. **Sampling randomness**: The specific batch `xb` might not perfectly represent the full training distribution

**But FID ≈ 6.6 is MUCH lower than FID ≈ 34 for DDPM-generated images!**

---

## Why Do This "Sanity Check"?

This test verifies that the FID metric is working correctly:

```
┌─────────────────────────────────────────────────────────────────┐
│                    SANITY CHECK RESULTS                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   Comparison                              Expected    Actual    │
│   ──────────────────────────────────────  ────────    ──────    │
│                                                                 │
│   Real (validation) vs Generated (DDPM)   HIGH        ~34      │
│   Real (validation) vs Real (training)    LOW         ~6.6     │
│   Real (validation) vs Real (validation)  ~ZERO       ~0       │
│                                                                 │
│   ✓ Generated images have HIGH FID (different from real)       │
│   ✓ Real images have LOW FID (similar to real)                 │
│   ✓ This confirms our FID metric is working correctly!         │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### What Would Be Wrong?

- If `ie.fid(xb)` returned ~34 (same as generated) → Something is broken!
- If `ie.fid(s)` returned ~6 (same as real) → DDPM is perfect (unlikely) or metric is broken

---

## The Code Flow

### Step 1: Create ImageEval (extracts validation features)

```python
ie = ImageEval(model, learn.dls, cbs=[DeviceCB()])

# Inside __init__:
# self.feats = features from dls.valid (10,000 validation images)
# self.stats = (mean, covariance) of validation features
```

### Step 2: Test with generated images

```python
ie.fid(s)  # s = DDPM-generated images
# Returns ~34 (high FID = generated images differ from real)
```

### Step 3: Sanity check with real images

```python
ie.fid(xb)  # xb = batch from TRAINING set (different from validation!)
# Returns ~6.6 (low FID = real images similar to real)
```

---

## Visual Summary

```
┌─────────────────────────────────────────────────────────────────┐
│                    FASHION MNIST DATASET                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   ┌─────────────────────────┐    ┌─────────────────────────┐   │
│   │     TRAINING SET        │    │    VALIDATION SET       │   │
│   │     (60,000 images)     │    │    (10,000 images)      │   │
│   │                         │    │                         │   │
│   │   xb = one batch        │    │   self.feats = ALL      │   │
│   │   (512 images)          │    │   (10,000 images)       │   │
│   │                         │    │                         │   │
│   └───────────┬─────────────┘    └───────────┬─────────────┘   │
│               │                              │                  │
│               │    DIFFERENT IMAGES!         │                  │
│               │                              │                  │
│               ▼                              ▼                  │
│        ┌─────────────────────────────────────────────┐         │
│        │         ie.fid(xb) compares these           │         │
│        │                                             │         │
│        │    Training batch  vs  Validation set       │         │
│        │    (512 images)       (10,000 images)       │         │
│        │                                             │         │
│        │              Result: FID ≈ 6.6              │         │
│        │         (low because both are real!)        │         │
│        └─────────────────────────────────────────────┘         │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## Comparison Table

| Test | What's Compared | FID Score | Interpretation |
|------|-----------------|-----------|----------------|
| `ie.fid(s)` | Validation vs DDPM-generated | ~34 | Generated images differ from real |
| `ie.fid(xb)` | Validation vs Training batch | ~6.6 | Real images similar to real (sanity check ✓) |
| `ie.kid(xb)` | Validation vs Training batch | ~-0.03 | Near zero (sanity check ✓) |

---

## Summary

| Question | Answer |
|----------|--------|
| What is `self.feats`? | Features from **validation set** (10,000 images) |
| What is `xb`? | One batch from **training set** (512 images) |
| Are they the same images? | **NO!** Different images from different splits |
| Why is FID ≈ 6.6 not zero? | Different splits + sample size + estimation variance |
| Why do this check? | Sanity check: real vs real should have LOW FID |
| What would be wrong? | If `ie.fid(xb)` returned ~34 (high), the metric is broken |

---

## Key Takeaway

**`ie.fid(xb)` is a sanity check** that compares:
- **Validation set features** (stored in `self.feats`)
- **Training batch features** (extracted from `xb`)

Both are real images, but from **different parts of the dataset**. The low FID (~6.6) confirms that:
1. Real images are similar to other real images ✓
2. The FID metric is working correctly ✓
3. Generated images (FID ~34) are noticeably different from real images ✓

---

## Section 7: Export

In [ ]:
# Export to miniai/fid.py
import nbdev
nbdev.nbdev_export()

---

## Summary

This notebook covered how to evaluate generated images using FID and KID.

---

**Key Concepts:**

1. **Feature Extraction**: Use a neural network to convert images to feature vectors
   - Features capture high-level image characteristics
   - Can use custom classifier or InceptionV3

2. **FID (Fréchet Inception Distance)**:
   - Assumes features are Gaussian distributed
   - Compares means and covariances
   - Formula: $FID = ||μ_1 - μ_2||^2 + Tr(Σ_1 + Σ_2 - 2\sqrt{Σ_1 Σ_2})$
   - Lower is better

3. **KID (Kernel Inception Distance)**:
   - No Gaussian assumption
   - Uses kernel MMD
   - Unbiased estimator
   - Lower is better

4. **ImageEval Class**: Convenient wrapper for evaluation

---

**Comparison:**

| Aspect | FID | KID |
|--------|-----|-----|
| Distribution assumption | Gaussian | None |
| Sample bias | Biased | Unbiased |
| Computation | Matrix square root | Kernel computations |
| Standard | Industry standard | Less common |

---

**Practical Tips:**

1. Use InceptionV3 for comparable scores with papers
2. Use custom classifier for domain-specific evaluation
3. Always report both FID and KID when possible
4. Use large sample sizes (>2000) for reliable FID